# IE643 - Week 4: knowledge distillation into the student (ablation runs)

**Version `distill-v7` (2026-10-23):** adds **sequence-level KD** (Kim & Rush 2016), run sets `seqkd1` and `seqkd2`. A10 (diagnosis prompts) was not adopted; even oracle prompts gave 0.74-0.91 positive findings/study (GT 1.79). Hypothesis: GT impressions are dominated by comparison language ("interval", "stable", "unchanged") that one image cannot predict, so CE's optimum is the most frequent template. The teacher's impressions describe only what is visible. Targets = the cached teacher impressions (`teacher_cache.jsonl`, route two_step), cleaned (markdown `**`/`#`, bracket tags, list numbers removed, lower-cased); empty or cap-hit teacher text falls back to GT. Train split only; val/test always scored against GT. **S1** = epochs 1-5 on teacher text, epochs 6-8 on GT with a 100-step re-warmup to half the peak LRs, then cosine to 0. **S2** = each example's target is GT or teacher text with p = 0.5, resampled every epoch. CE only, prefix, 384 px, two seeds each. **Pre-registered rule (per method, both seeds):** paired TEST CE14-F1 CI of (run - A0) lower bound > -0.02, CE5-F1 > 0.226 (CE-only seed mean 0.188 + 3 sd), hit-cap <= 10%. Plan: `claude/v7_seqkd_plan.md`.

**Version `distill-v6.1` (2026-10-23):** smoke runs now write to `distill_runs_smoke/` and are never imported; the import also skips any run whose config says it was a smoke/CPU run (v6 imported the smoke A10 as finished and the evaluation crashed).

**Version `distill-v6` (2026-10-23):** adds run set `prompt` = **A10 diagnosis-driven prompts** (idea from PromptMRG, Jin et al. 2024; design ours). Ablation 2 showed the clinical head detects findings that the generated text omits (A6 head CE5-F1 0.518 vs its text 0.210). A10 = A6 (CE + clinical BCE, prefix, 384 px) + 14 prompt tokens, one per CheXpert label, each = LayerNorm(label embedding + state embedding), state = positive / not positive, inserted between the 32 visual prefix tokens and BOS. Training: per study, the prompt states come from the GT labels (CSV, positive = 1) with probability `p_gt` = 0.5, otherwise from the head's own detached predictions. Inference: head predictions at per-label thresholds chosen on VAL (max F1 vs CheXbert(GT impression)), recomputed after every epoch and saved with the weights. Extra TEST decodes (reported only): oracle prompts (CheXbert(GT impression) labels) and all-negative prompts. **Pre-registered rule:** adopt A10 if, for both seeds (A10, A10s1), the paired CE14-F1 CI of (A10 - A0) is entirely above 0 and hit-cap <= 10%.

**Version `distill-v5`** (v4.1 + new `ablation2`: CE-only seed repeats, resolution and connector ablations with the final loss, and clinical-head scoring). Ablation 3 settled the loss: token KD dropped (A9 failed its pre-registered rule), clinical and feature terms showed no CE14-F1 gain, so the final loss is **CE only**. `ablation2` = A4 CE xattn, A0s1 / A0s2 CE with seeds 1 and 2 (seed spread for every comparison), A5 CE at 224 px; attach this notebook's **ablation-3 output** (it contains A0-A9). Pre-registered: keep `prefix` unless the paired CE14-F1 CI of (A4 - A0) is entirely above 0; adopt 224 px only if the paired CI of (A5 - A0) has its lower bound above -0.02 (non-inferiority; 224 is cheaper on device). Earlier version notes: v4.1 = v4 + run A9, masked token KD. `ablation3` = the non-token-KD terms without token KD (A6-A8) plus A9, a token-KD run whose targets are cleaned using what `ie643_kd_diagnostic` found: the teacher's top-1 is `[` at the first token of all 7,996 train studies, `[`/EOS/`**` at 66% of sentence starts, and the GT token is absent from its top-20 there. Trains the student with the composite objective and runs the ablation table, two configs at a time (one per T4).

$$\mathcal L = \alpha\,\mathrm{CE}(\text{GT}) + \beta\,\tau^2\,\mathrm{KL}_{\text{top-}k}(p_T\,\|\,p_S) + \gamma\,\mathrm{BCE}_{\text{clin}} + \delta\,(1-\cos)_{\text{feat}}$$

| term | data | where the technique comes from |
|---|---|---|
| CE | ground-truth impression | standard teacher-forced LM loss (alpha = 1 always) |
| soft-target KD, tau^2 scaling | `teacher_topk.npz` (teacher-forced on the GT impression) | Hinton, Vinyals & Dean 2015; per-token ("word-level") KD for sequences: Kim & Rush 2016; tau = 2 as in DistilBERT/DistilGPT2 (Sanh et al. 2019) |
| top-k storage and its bias | top-20 per token, ids >= 50257 dropped | Anshumann et al. 2025 (Sparse Logit Sampling): renormalised top-k is biased and with K < 25 was worse than CE in their LM-pretraining setting; A1t1 vs A1r tests their residual-to-ground-truth correction at the same tau = 1 |
| per-term gradient norms | logged every 100 steps | GradNorm (Chen et al. 2018) uses per-task gradient norms to judge task balance; here only measured, not used to set weights |
| feature KD, 1 - cosine | `teacher_feats.npz` through linear heads | hint-based feature distillation (FitNets, Romero et al. 2015) with the cosine loss of DistilBERT; per-dimension target normalisation as for MAE's normalised targets (He et al. 2022), checked by the printed diagnostic |
| clinical BCE head | 14 CSV labels, positive = 1, uncertain ignored | multi-label finding classification alongside report generation (Jing et al. 2018; PromptMRG, Jin et al. 2024); "U-Ignore" for uncertain labels (CheXpert, Irvin et al. 2019); pos_weight for imbalance |
| align then full stages | | LLaVA (Liu et al. 2023) two-stage training |

**Run sets** (`RUN_SET`): `ablation1` = A0 CE, A1 +KD (renormalised top-k, tau 2), A1t1 +KD (renormalised top-k, tau 1), A1r +KD (residual-to-GT, tau 1), A2 +KD+feat, A3 full. A1 vs A0: does KD help; A1t1 vs A1: temperature; A1r vs A1t1: the bias correction. `ablation2` = A4 full with the other connector (xattn), A5 full at 224 px. `ablation3` = A6 CE + clinical BCE, A7 CE + feature KD, A8 CE + clinical + feature (no token KD), A9 CE + masked token KD at tau 1 (positions kept only if the GT token is in the teacher top-20 and the teacher top-1 is neither a formatting token nor EOS; formatting tokens and EOS removed from the target, which is renormalised); attach this notebook's ablation-1 output so A0 is imported for the paired comparison. **Decision rule for A9 (fixed before the run):** keep token KD only if the paired CE14-F1 CI of (A9 - A0) is entirely above 0 and A9's cap-hit rate stays near A0's; otherwise drop token KD and report it as a negative result.
**Epoch selection:** every epoch from `EVAL_FROM` on, all val images are greedily decoded and scored with CheXbert (14-label micro-F1 vs CheXbert on the GT impression); the best epoch is kept. Test is decoded once, with the best epoch.
**Collapse diagnostics** (the pilot produced only 22-34 distinct impressions for 999 val studies): share of the most common output, positive findings per study vs ground truth, No Finding rate.

**How to run**
1. `MODE = "smoke"`, Accelerator **None**, Internet **On** -> Save Version -> Save & Run All (Commit). Free, ~10 min. Last line must be `SMOKE OK`.
2. `MODE = "train"`, `RUN_SET = "ablation1"` (`CONNECTOR` is already `prefix` from the pilot), Accelerator **GPU T4 x2** -> Save & Run All (Commit).
3. `RUN_SET = "ablation2"`: Edit the committed version, **attach its own output** (finished runs are imported and shown in the same table), commit again.

**Inputs to attach:** notebook outputs **IE643 Cleaning Step 7** (`reports_clean.csv`), **IE643 Cleaning** (`images_512.npy`), **IE643 teacher cache** merge version (`teacher_final/`).

In [ ]:
import time
T0 = time.time()
NOTEBOOK_VERSION = "distill-v7"
MODE = "smoke"                 # "smoke" (CPU, free) -> "train" (GPU T4 x2, committed)
RUN_SET = "seqkd1"          # "seqkd1" (S1, S1s1: teacher text -> GT) | "seqkd2" (S2, S2s1: mixed) | "prompt" (A10) | "ablation1" | "ablation3" | "ablation2" | "ablation2_fullloss" (old) | "custom"
CONNECTOR = "prefix"           # pilot-v1 decision (2026-10-21): prefix, paired CI of CE14-F1(prefix - xattn) [+0.027, +0.057]

INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
CSV_PATH = NPY_PATH = TEACHER_DIR = None       # None = find the attached inputs automatically

ENCODER_384, ENCODER_224 = "vit_tiny_patch16_384.augreg_in21k_ft_in1k", "vit_tiny_patch16_224.augreg_in21k_ft_in1k"
EPOCHS, ALIGN_EPOCHS, EVAL_FROM = 8, 1, 2      # epoch 1 = align; val generation + CheXbert from epoch EVAL_FROM on
BATCH = 16
LR_NEW, LR_PRE, WD, WARMUP, CLIP = 2e-4, 3e-5, 0.01, 200, 1.0
MAX_LEN = None                 # None = p95 of tokenised train impression length + 1 (same rule as the pilot), capped at 256
# loss weights for the terms that are switched on (alpha = 1 always)
BETA, TAU, DELTA, GAMMA = 0.5, 2.0, 0.5, 0.5
POS_WEIGHT_MAX = 10.0
P_TEACHER, SEQKD_GT_FROM = 0.5, 6   # S2: P(teacher-text target); S1: first epoch trained on GT
P_GT = 0.5                     # A10: probability that a training study's prompt states come from the GT labels (else from the head)
SEED = 0
SAVE_WEIGHTS = True            # best-epoch fp16 weights per run (~200 MB each)
BUDGET_H = 11.0                # no new run starts if it would not finish before this wall time (12 h session limit)
EST_RUN_H = 0.7                # used for that check until a run on the same GPU has finished and been timed (pilot: 2.2-2.4 min/epoch)
PRETRAINED = True              # keep True on Kaggle
OFFLINE_TEST = False           # only for testing this notebook outside Kaggle (fake tokenizer / labeler)
CUSTOM_RUNS = {"A5_ce_224": dict(encoder=ENCODER_224, img=224)}               # e.g. {"A6_full_tau1": dict(beta=BETA, delta=DELTA, gamma=GAMMA, tau=1.0)}
print("notebook version:", NOTEBOOK_VERSION, "| MODE =", MODE, "| RUN_SET =", RUN_SET, "| CONNECTOR =", CONNECTOR)

In [ ]:
MODEL_PY = r'''"""IE643 student: ViT-Tiny encoder -> connector -> DistilGPT2 decoder (+ distillation / clinical heads).

Two ways to condition the decoder on the image (cfg.mode):
  "xattn"  : cross-attention inside every DistilGPT2 block (CvT2DistilGPT2 style). Main model.
  "prefix" : a Perceiver-style resampler turns the patch tokens into `n_prefix` vectors that are
             prepended to the text embeddings (ClipCap / LLaVA style). Ablation. With
             set_stage("align") kept for the whole run this is the frozen-LM prefix-tuning ablation.

Conventions
  * BOS == EOS == 50256 (GPT-2 has no BOS). Student vocab = 50257, teacher ids >= 50257 are dropped in the loss.
  * forward() returns logits ALIGNED with labels: logits[:, j] predicts labels[:, j]
    (input = [BOS, t0..t_{n-1}], labels = [t0..t_{n-1}, EOS]). No internal shift anywhere.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel

EOS_ID = 50256
VOCAB = 50257


@dataclass
class StudentConfig:
    # encoder
    # Main setting: native-384 checkpoint (576 patch tokens, 24x24 grid -> clean 4x4 pooling).
    # 224 ablation: encoder_name="vit_tiny_patch16_224.augreg_in21k_ft_in1k", img_size=224.
    # Keep encoder_name/img_size identical to the NN-retrieval baseline you compare against.
    encoder_name: str = "vit_tiny_patch16_384.augreg_in21k_ft_in1k"
    img_size: int = 384
    pretrained_enc: bool = True
    # decoder
    decoder_name: str = "distilgpt2"
    pretrained_dec: bool = True
    mode: str = "xattn"            # "xattn" | "prefix"
    n_prefix: int = 32             # prefix mode: number of visual prefix tokens
    resampler_heads: int = 8
    # distillation targets (cached teacher features)
    d_grid: int = 1024             # siglip_grid4  [16, 1024]
    d_mean: int = 1024             # siglip_mean   [1024]
    d_proj: int = 2560             # proj_mean     [2560]
    grid_hw: int = 4
    # clinical head
    n_clin: int = 14
    diag_prompt: bool = False      # A10: head predictions -> 14 prompt tokens between the visual prefix and BOS (prefix mode only)
    drop: float = 0.1
    max_len: int = 192             # max text tokens incl. BOS. EDA: median impression 86 tok, p95 187 (pre-cleaning);
                                   # set from the p95 of ntok_final in reports_clean.csv before training.


class Connector(nn.Module):
    """Linear(D_v->H) -> GELU -> Linear(H->H) -> LayerNorm (LLaVA-1.5 style 2-layer MLP)."""

    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.LayerNorm(d_out)

    def forward(self, x):
        return self.norm(self.net(x))


class Resampler(nn.Module):
    """m learned queries cross-attend to the visual tokens -> [B, m, H] (Perceiver / Q-Former lite)."""

    def __init__(self, d: int, n_queries: int, n_heads: int, drop: float = 0.0):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(n_queries, d) * 0.02)
        self.ln_q, self.ln_kv, self.ln_ff = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, kv):
        q = self.queries.unsqueeze(0).expand(kv.size(0), -1, -1)
        kv = self.ln_kv(kv)
        x = q + self.attn(self.ln_q(q), kv, kv, need_weights=False)[0]
        return x + self.ff(self.ln_ff(x))


class XRayReportStudent(nn.Module):
    def __init__(self, cfg: StudentConfig):
        super().__init__()
        assert cfg.mode in ("xattn", "prefix")
        self.cfg = cfg

        # ---- encoder -------------------------------------------------------------------------
        self.encoder = timm.create_model(
            cfg.encoder_name, pretrained=cfg.pretrained_enc, num_classes=0, img_size=cfg.img_size
        )
        self.n_prefix_tok = getattr(self.encoder, "num_prefix_tokens", 1)  # CLS (+ reg tokens)
        d_v = self.encoder.num_features
        self.d_v = d_v
        self.enc_cfg = self.encoder.pretrained_cfg  # mean/std for preprocessing

        # ---- decoder -------------------------------------------------------------------------
        xattn = cfg.mode == "xattn"
        if cfg.pretrained_dec:
            self.decoder = GPT2LMHeadModel.from_pretrained(cfg.decoder_name, add_cross_attention=xattn)
        else:  # offline / unit-test path: distilgpt2 geometry, random init
            gcfg = GPT2Config(n_layer=6, n_head=12, n_embd=768, vocab_size=VOCAB, n_positions=1024,
                              add_cross_attention=xattn)
            self.decoder = GPT2LMHeadModel(gcfg)
        self.d_t = self.decoder.config.n_embd

        # ---- connector -----------------------------------------------------------------------
        self.connector = Connector(d_v, self.d_t)
        self.resampler = (
            Resampler(self.d_t, cfg.n_prefix, cfg.resampler_heads, cfg.drop) if cfg.mode == "prefix" else None
        )

        # ---- auxiliary heads (train only; dropped at export) --------------------------------
        self.grid_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_grid))
        self.mean_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_mean))
        self.proj_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_proj))
        self.clin_head = nn.Sequential(
            nn.LayerNorm(2 * d_v), nn.Dropout(cfg.drop), nn.Linear(2 * d_v, d_v), nn.GELU(),
            nn.Linear(d_v, cfg.n_clin),
        )

        # ---- diagnosis prompts (A10): token_c = LayerNorm(label_c + state[s_c]), s_c in {0 = not positive, 1 = positive} ---
        if cfg.diag_prompt:
            assert cfg.mode == "prefix", "diagnosis prompts are implemented for the prefix connector only"
            self.prompt_label = nn.Parameter(torch.randn(cfg.n_clin, self.d_t) * 0.02)
            self.prompt_state = nn.Parameter(torch.randn(2, self.d_t) * 0.02)
            self.prompt_ln = nn.LayerNorm(self.d_t)
            # per-label decision thresholds of the head, set on VAL by the worker; a buffer so they are saved with the weights
            self.register_buffer("prompt_thr", torch.full((cfg.n_clin,), 0.5))

    # ------------------------------------------------------------------------------------------
    def encode(self, images):
        """images [B,3,H,W] (already normalised) -> dict of visual tokens and head predictions."""
        feats = self.encoder.forward_features(images)              # [B, npt+N, D]
        cls = feats[:, 0]
        patches = feats[:, self.n_prefix_tok:]                     # [B, N, D]
        B, N, D = patches.shape
        g = int(math.isqrt(N))
        assert g * g == N, "patch tokens must form a square grid"
        mean = patches.mean(1)
        grid = F.adaptive_avg_pool2d(patches.transpose(1, 2).reshape(B, D, g, g), self.cfg.grid_hw)
        grid = grid.flatten(2).transpose(1, 2)                     # [B, 16, D] row-major
        memory = self.connector(feats)                             # [B, npt+N, H]
        if self.resampler is not None:
            memory = self.resampler(memory)                        # [B, m, H]
        return {"memory": memory, "cls": cls, "mean": mean, "grid": grid}

    def heads(self, enc):
        return {
            "grid_pred": self.grid_head(enc["grid"]),              # [B,16,d_grid]
            "mean_pred": self.mean_head(enc["mean"]),              # [B,d_mean]
            "proj_pred": self.proj_head(enc["mean"]),              # [B,d_proj]
            "clin_logits": self.clin_head(torch.cat([enc["cls"], enc["mean"]], -1)),  # [B,n_clin]
        }

    def clin_probs(self, enc):
        return torch.sigmoid(self.clin_head(torch.cat([enc["cls"], enc["mean"]], -1)).float())   # [B, n_clin] fp32

    def prompt_embeds(self, states):
        """states [B, n_clin] in {0,1} -> prompt tokens [B, n_clin, H]."""
        s = states.long().clamp(0, 1)
        return self.prompt_ln(self.prompt_label.unsqueeze(0) + self.prompt_state[s])

    def head_states(self, enc):
        return (self.clin_probs(enc).detach() >= self.prompt_thr).long()

    def forward(self, images, input_ids, attention_mask=None, with_heads=True, prompt_states=None, prompt_gt=None, p_gt=0.0):
        """prompt_states: explicit [B, n_clin] states (diag_prompt only); otherwise the head's own thresholded predictions,
        replaced per study by prompt_gt with probability p_gt (training)."""
        enc = self.encode(images)
        mem = enc["memory"]
        hd = self.heads(enc) if (with_heads or self.cfg.diag_prompt) else {}
        if self.cfg.diag_prompt:
            if prompt_states is None:
                prompt_states = (torch.sigmoid(hd["clin_logits"].detach().float()) >= self.prompt_thr).long()
                if prompt_gt is not None and p_gt > 0:
                    use_gt = torch.rand(prompt_states.size(0), 1, device=prompt_states.device) < p_gt
                    prompt_states = torch.where(use_gt, (prompt_gt > 0.5).long(), prompt_states)
            mem = torch.cat([mem, self.prompt_embeds(prompt_states).to(mem.dtype)], 1)
        if self.cfg.mode == "xattn":
            out = self.decoder(input_ids=input_ids, attention_mask=attention_mask,
                               encoder_hidden_states=mem, use_cache=False)
            logits = out.logits
        else:
            m = mem.size(1)
            emb = torch.cat([mem, self.decoder.transformer.wte(input_ids)], 1)
            am = None
            if attention_mask is not None:
                am = torch.cat([attention_mask.new_ones(attention_mask.size(0), m), attention_mask], 1)
            out = self.decoder(inputs_embeds=emb, attention_mask=am, use_cache=False)
            logits = out.logits[:, m:]                              # drop prefix positions
        res = {"logits": logits}
        if with_heads:
            res.update(hd)
        return res

    # ------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, images, max_new_tokens=60, num_beams=3, no_repeat_ngram_size=3,
                 length_penalty=1.0, **kw):
        """Beam (or greedy if num_beams=1) decoding with KV cache. Returns list[list[int]] without BOS/EOS."""
        assert not self.cfg.diag_prompt, "use greedy() for diagnosis-prompt models"
        was_training = self.training
        self.eval()
        enc = self.encode(images)
        mem = enc["memory"]
        B = mem.size(0)
        bos = torch.full((B, 1), EOS_ID, dtype=torch.long, device=mem.device)
        common = dict(max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False,
                      no_repeat_ngram_size=no_repeat_ngram_size, length_penalty=length_penalty,
                      eos_token_id=EOS_ID, pad_token_id=EOS_ID, early_stopping=True, **kw)
        if self.cfg.mode == "xattn":
            ids = self.decoder.generate(input_ids=bos, encoder_hidden_states=mem, **common)
            seqs = ids[:, 1:]                                       # strip BOS
        else:
            emb = torch.cat([mem, self.decoder.transformer.wte(bos)], 1)
            seqs = self.decoder.generate(inputs_embeds=emb, **common)  # returns only new tokens
        out = []
        for s in seqs.tolist():
            if EOS_ID in s:
                s = s[: s.index(EOS_ID)]
            out.append(s)
        self.train(was_training)
        return out

    @torch.no_grad()
    def greedy(self, images, max_new_tokens=128):
        """Greedy decoding with the decoder's KV cache, written by hand so it does not depend on the
        transformers version's generate() handling of encoder_hidden_states / inputs_embeds.
        Returns list[list[int]] without BOS/EOS."""
        was_training = self.training
        self.eval()
        mem = self.encode(images)["memory"]
        B = mem.size(0)
        cur = torch.full((B, 1), EOS_ID, dtype=torch.long, device=mem.device)
        if self.cfg.mode == "xattn":
            out = self.decoder(input_ids=cur, encoder_hidden_states=mem, use_cache=True)
        else:
            out = self.decoder(inputs_embeds=torch.cat([mem, self.decoder.transformer.wte(cur)], 1), use_cache=True)
        seqs = []
        done = torch.zeros(B, dtype=torch.bool, device=mem.device)
        for _ in range(max_new_tokens):
            nxt = out.logits[:, -1, :VOCAB].argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            seqs.append(nxt)
            done |= nxt == EOS_ID
            if bool(done.all()):
                break
            kw = dict(input_ids=nxt[:, None], past_key_values=out.past_key_values, use_cache=True)
            if self.cfg.mode == "xattn":
                kw["encoder_hidden_states"] = mem
            out = self.decoder(**kw)
        res = []
        for s in torch.stack(seqs, 1).tolist():
            res.append(s[: s.index(EOS_ID)] if EOS_ID in s else s)
        self.train(was_training)
        return res

    @torch.no_grad()
    def greedy(self, images, max_new_tokens=128, prompt_states=None):
        """Greedy decoding with an explicit KV-cache loop (does not rely on HF generate(), whose handling of
        encoder_hidden_states for decoder-only GPT-2 differs across transformers versions).
        Returns (list[list[int]] without BOS/EOS, list[bool] hit_cap = no EOS within max_new_tokens)."""
        was_training = self.training
        self.eval()
        enc = self.encode(images)
        mem = enc["memory"]
        if self.cfg.diag_prompt:
            st = prompt_states if prompt_states is not None else self.head_states(enc)
            mem = torch.cat([mem, self.prompt_embeds(st).to(mem.dtype)], 1)
        B, dev = mem.size(0), mem.device
        xattn = self.cfg.mode == "xattn"
        bos = torch.full((B, 1), EOS_ID, dtype=torch.long, device=dev)
        if xattn:
            out = self.decoder(input_ids=bos, encoder_hidden_states=mem, use_cache=True)
        else:
            out = self.decoder(inputs_embeds=torch.cat([mem, self.decoder.transformer.wte(bos)], 1), use_cache=True)
        past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        steps = []
        for step in range(max_new_tokens):
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            steps.append(nxt)
            done = done | (nxt == EOS_ID)
            if bool(done.all()) or step == max_new_tokens - 1:
                break
            kw = dict(input_ids=nxt[:, None], past_key_values=past, use_cache=True)
            if xattn:
                kw["encoder_hidden_states"] = mem
            out = self.decoder(**kw)
            past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        toks = torch.stack(steps, 1).tolist()
        seqs, capped = [], []
        for s in toks:
            if EOS_ID in s:
                seqs.append(s[: s.index(EOS_ID)]); capped.append(False)
            else:
                seqs.append(s); capped.append(True)
        self.train(was_training)
        return seqs, capped

    # ------------------------------------------------------------------------------------------
    def _is_pretrained(self, name: str) -> bool:
        if name.startswith("encoder."):
            return True
        return name.startswith("decoder.") and "crossattn" not in name and "ln_cross_attn" not in name

    def set_stage(self, stage: str):
        """'align': freeze pretrained encoder+decoder, train connector / cross-attn / resampler / heads.
        'full'  : train everything (use a lower LR on pretrained parts via param_groups)."""
        assert stage in ("align", "full")
        for n, p in self.named_parameters():
            p.requires_grad = True if stage == "full" else not self._is_pretrained(n)

    def param_groups(self, lr_new=1e-4, lr_pre=3e-5, wd=0.01):
        buckets = {(pre, dec): [] for pre in (True, False) for dec in (True, False)}
        for n, p in self.named_parameters():
            if p.requires_grad:
                buckets[(self._is_pretrained(n), p.ndim >= 2 and "wpe" not in n)].append(p)
        groups = []
        for (pre, dec), ps in buckets.items():
            if ps:
                groups.append({"params": ps, "lr": lr_pre if pre else lr_new, "weight_decay": wd if dec else 0.0})
        return groups

    def size_report(self):
        def cnt(m):
            return sum(p.numel() for p in m.parameters())
        wte = self.decoder.transformer.wte.weight.numel()
        r = {
            "encoder": cnt(self.encoder),
            "connector": cnt(self.connector) + (cnt(self.resampler) if self.resampler is not None else 0),
            "decoder_total": cnt(self.decoder),
            "decoder_token_embedding": wte,
            "train_only_heads": sum(cnt(m) for m in (self.grid_head, self.mean_head, self.proj_head, self.clin_head)),
        }
        r["deploy_total"] = r["encoder"] + r["connector"] + r["decoder_total"]
        if self.cfg.diag_prompt:   # the head and the prompt tokens are part of the deployed graph for A10
            r["clin_head_and_prompts_deployed"] = cnt(self.clin_head) + self.prompt_label.numel() + self.prompt_state.numel() + cnt(self.prompt_ln)
            r["deploy_total"] += r["clin_head_and_prompts_deployed"]
        r["deploy_total_after_vocab_prune_to_8k"] = r["deploy_total"] - wte + 8000 * self.d_t
        return r
'''

LOSSES_PY = r'''"""IE643 composite distillation loss: alpha*CE + beta*tau^2*KL_topk + gamma*BCE_clin + delta*(1-cos) features.

Batch fields consumed (built by the training worker's collate fn):
  labels        [B,L]   long, -100 on padding; labels[:, j] is predicted by logits[:, j]
  kd_ids        [B,L,K] long   teacher top-k token ids at each GT-impression position (pad with 0)
  kd_logp       [B,L,K] float  teacher top-k log-probs at tau=1
  kd_mask       [B,L]   bool   True where a teacher row exists for this position
  kd_weight     [B]     float  optional per-example weight
  clin_y        [B,C]   float  1 = positive, 0 = otherwise
  clin_m        [B,C]   float  1 = this label counts in the loss, 0 = ignore
  t_grid [B,16,1024], t_mean [B,1024], t_proj [B,2560], feat_mask [B]  cached SigLIP / projector features
"""
from __future__ import annotations

import torch
import torch.nn.functional as F

VOCAB = 50257


def ce_loss(logits, labels):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), ignore_index=-100)


def topk_kd_loss(logits, kd_ids, kd_logp, pos_mask, tau=2.0, sample_weight=None, vocab=VOCAB):
    """Forward KL on the teacher's top-k support.

    Teacher: drop ids >= vocab (CheXagent-only tokens), apply temperature, renormalise over what is left.
    Student: full-vocab log-softmax at the same temperature, gathered at the teacher ids
             (so student mass outside the teacher's support is also penalised).
    pos_mask [B,L] selects the positions that take part (teacher row exists and label is not padding).
    """
    logits = logits[..., :vocab].float()
    logq = F.log_softmax(logits / tau, dim=-1)
    valid = (kd_ids >= 0) & (kd_ids < vocab)
    lp = kd_logp.float().masked_fill(~valid, -1e9)
    pt = F.softmax(lp / tau, dim=-1) * valid
    pt = pt / pt.sum(-1, keepdim=True).clamp_min(1e-12)
    logq_g = logq.gather(-1, kd_ids.clamp(0, vocab - 1))
    kl = (pt * (pt.clamp_min(1e-12).log() - logq_g) * valid).sum(-1)       # [B,L]
    m = (pos_mask & valid.any(-1)).float()
    if sample_weight is not None:
        m = m * sample_weight.float().unsqueeze(1)
    return (tau ** 2) * (kl * m).sum() / m.sum().clamp_min(1.0)


def kd_keep_mask(kd_ids, kd_logp, labels, pos_mask, style, vocab=VOCAB, eos=50256):
    """Positions where the teacher target is usable: the GT token is in the teacher top-k AND the teacher's top-1 is
    neither a formatting token (markdown, list brackets, newline, bare list number) nor EOS.
    Motivation (ie643_kd_diagnostic, train split): at the first token the teacher top-1 is '[' for 100% of studies and the GT
    token is never in its top-20; at sentence starts the top-1 is '[' / EOS / '**' for 66% and GT is in the top-20 for 55%;
    inside sentences top-1 is a formatting token for 0.4% and GT is in the top-20 for 96%.
    style: bool tensor [vocab] (True = formatting token). Returns bool [B,L]."""
    valid = (kd_ids >= 0) & (kd_ids < vocab)
    lp = kd_logp.float().masked_fill(~valid, -1e9)
    top1 = kd_ids.gather(-1, lp.argmax(-1, keepdim=True)).squeeze(-1).clamp(0, vocab - 1)
    top1_bad = style.to(kd_ids.device)[top1] | (top1 == eos)
    gt_in = ((kd_ids == labels.unsqueeze(-1)) & valid).any(-1)
    return pos_mask & gt_in & ~top1_bad & valid.any(-1)


def topk_kd_masked_loss(logits, kd_ids, kd_logp, labels, pos_mask, style, tau=1.0, sample_weight=None, vocab=VOCAB, eos=50256):
    """Top-k KD restricted to the positions of kd_keep_mask, with formatting tokens and EOS removed from the teacher target
    (the rest renormalised). Student side as in topk_kd_loss. Returns (loss, kept_fraction_of_label_positions)."""
    keep = kd_keep_mask(kd_ids, kd_logp, labels, pos_mask, style, vocab, eos)
    drop = style.to(kd_ids.device)[kd_ids.clamp(0, vocab - 1)] | (kd_ids == eos)
    logits = logits[..., :vocab].float()
    logq = F.log_softmax(logits / tau, dim=-1)
    valid = (kd_ids >= 0) & (kd_ids < vocab) & ~drop
    lp = kd_logp.float().masked_fill(~valid, -1e9)
    pt = F.softmax(lp / tau, dim=-1) * valid
    pt = pt / pt.sum(-1, keepdim=True).clamp_min(1e-12)
    logq_g = logq.gather(-1, kd_ids.clamp(0, vocab - 1))
    kl = (pt * (pt.clamp_min(1e-12).log() - logq_g) * valid).sum(-1)
    m = (keep & valid.any(-1)).float()
    kept = m.sum() / (pos_mask & (labels != -100)).float().sum().clamp_min(1.0)
    if sample_weight is not None:
        m = m * sample_weight.float().unsqueeze(1)
    return (tau ** 2) * (kl * m).sum() / m.sum().clamp_min(1.0), kept


def topk_kd_residual_loss(logits, kd_ids, kd_logp, labels, pos_mask, sample_weight=None, vocab=VOCAB):
    """Top-k KD with the teacher's residual mass given to the ground-truth token (tau = 1).

    Anshumann et al. 2025 (Sparse Logit Sampling, ACL) show that renormalised top-k targets are a biased estimate
    of the teacher distribution; assigning the residual mass 1 - sum(top-k) to the ground-truth token is one of the
    corrections they evaluate. Teacher-only ids (>= vocab) are dropped and their mass goes to the residual too.
    The target sums to 1 without renormalisation, so it only works at temperature 1.
    """
    logq = F.log_softmax(logits[..., :vocab].float(), dim=-1)
    valid = (kd_ids >= 0) & (kd_ids < vocab)
    p = kd_logp.float().exp() * valid
    r = (1.0 - p.sum(-1)).clamp_min(0.0)                                   # [B,L]
    lab = labels.clamp(0, vocab - 1)
    in_sup = (kd_ids == lab.unsqueeze(-1)) & valid
    p = p + in_sup.float() * r.unsqueeze(-1)
    extra = r * (~in_sup.any(-1)).float()
    ids2 = torch.cat([kd_ids.clamp(0, vocab - 1), lab.unsqueeze(-1)], -1)
    p2 = torch.cat([p, extra.unsqueeze(-1)], -1)
    logq_g = logq.gather(-1, ids2)
    kl = (p2 * (p2.clamp_min(1e-12).log() - logq_g)).sum(-1)
    m = (pos_mask & valid.any(-1)).float()
    if sample_weight is not None:
        m = m * sample_weight.float().unsqueeze(1)
    return (kl * m).sum() / m.sum().clamp_min(1.0)


def feat_loss(pred, target, sample_mask=None):
    """1 - cosine similarity, averaged over (samples, cells)."""
    cos = F.cosine_similarity(pred.float(), target.float(), dim=-1)
    loss = 1.0 - cos
    if loss.ndim > 1:
        loss = loss.mean(dim=tuple(range(1, loss.ndim)))
    if sample_mask is not None:
        w = sample_mask.float()
        return (loss * w).sum() / w.sum().clamp_min(1.0)
    return loss.mean()


def clin_loss(logits, y, m, pos_weight=None):
    l = F.binary_cross_entropy_with_logits(logits.float(), y.float(), pos_weight=pos_weight, reduction="none")
    m = m.float()
    return (l * m).sum() / m.sum().clamp_min(1.0)


class DistillLoss:
    def __init__(self, alpha=1.0, beta=0.5, gamma=0.2, delta=0.2, tau=2.0, pos_weight=None,
                 w_grid=1.0, w_mean=1.0, w_proj=1.0, kd_residual=False, kd_masked=False, style_mask=None):
        self.alpha, self.beta, self.gamma, self.delta, self.tau = alpha, beta, gamma, delta, tau
        self.kd_residual = kd_residual
        self.kd_masked, self.style_mask = kd_masked, style_mask
        assert not kd_residual or tau == 1.0, "residual-to-GT top-k KD is defined at temperature 1"
        assert not kd_masked or style_mask is not None, "masked KD needs the formatting-token mask (data file key style_mask)"
        assert not (kd_masked and kd_residual), "pick one KD variant"
        self.pos_weight = pos_weight          # tensor [C] from training label prevalence, or None
        self.w_feat = (w_grid, w_mean, w_proj)

    def __call__(self, out, batch):
        labels = batch["labels"]
        parts = {}
        self.terms = {}                       # weighted loss tensors (graph kept) for the gradient-norm probe
        total = out["logits"].new_zeros((), dtype=torch.float32)
        if self.alpha:
            parts["ce"] = ce_loss(out["logits"], labels)
            self.terms["ce"] = self.alpha * parts["ce"]
            total = total + self.terms["ce"]
        if self.beta and "kd_ids" in batch:
            pos = batch["kd_mask"] & (labels != -100)
            if self.kd_masked:
                kd, kept = topk_kd_masked_loss(out["logits"], batch["kd_ids"], batch["kd_logp"], labels, pos, self.style_mask,
                                               self.tau, batch.get("kd_weight"))
                parts["kd"], parts["kd_kept"] = kd, kept
            elif self.kd_residual:
                parts["kd"] = topk_kd_residual_loss(out["logits"], batch["kd_ids"], batch["kd_logp"], labels, pos,
                                                    batch.get("kd_weight"))
            else:
                parts["kd"] = topk_kd_loss(out["logits"], batch["kd_ids"], batch["kd_logp"], pos,
                                           self.tau, batch.get("kd_weight"))
            self.terms["kd"] = self.beta * parts["kd"]
            total = total + self.terms["kd"]
        if self.gamma and "clin_y" in batch:
            pw = self.pos_weight.to(out["clin_logits"].device) if self.pos_weight is not None else None
            parts["clin"] = clin_loss(out["clin_logits"], batch["clin_y"], batch["clin_m"], pw)
            self.terms["clin"] = self.gamma * parts["clin"]
            total = total + self.terms["clin"]
        if self.delta and "t_grid" in batch:
            fm = batch.get("feat_mask")
            wg, wm, wp = self.w_feat
            parts["f_grid"] = feat_loss(out["grid_pred"], batch["t_grid"], fm)
            parts["f_mean"] = feat_loss(out["mean_pred"], batch["t_mean"], fm)
            parts["f_proj"] = feat_loss(out["proj_pred"], batch["t_proj"], fm)
            parts["feat"] = (wg * parts["f_grid"] + wm * parts["f_mean"] + wp * parts["f_proj"]) / (wg + wm + wp)
            self.terms["feat"] = self.delta * parts["feat"]
            total = total + self.terms["feat"]
        parts["total"] = total
        return total, {k: float(v.detach()) for k, v in parts.items()}
'''

DISTILL_WORKER_PY = r'''"""IE643 distillation worker: trains ONE student config with the composite loss, selects the best epoch on
val CheXbert F1 of greedily generated impressions, then writes val + test predictions.

Usage: python distill_worker.py <cfg.json> <run_name> <out_dir>
One process per GPU (CUDA_VISIBLE_DEVICES set by the notebook), or CPU for the smoke test.
Writes to out_dir: worker.log, curve.jsonl, summary.json, pred_val.csv, pred_test.csv, best.pt (fp16, optional).
"""
import json, math, os, random, sys, time

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

cfg_path, RUN, OUT = sys.argv[1], sys.argv[2], sys.argv[3]
G = json.load(open(cfg_path))
C = dict(G["defaults"], **G["runs"][RUN])             # run settings override the shared defaults
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import StudentConfig, XRayReportStudent, EOS_ID, VOCAB  # noqa: E402
from losses import DistillLoss                                     # noqa: E402

LOG = open(f"{OUT}/worker.log", "a", buffering=1)
T0 = time.time()


def log(*a):
    s = f"[{RUN} {time.time() - T0:7.0f}s] " + " ".join(str(x) for x in a)
    print(s, flush=True)
    LOG.write(s + "\n")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


seed_all(C["seed"])
dev = torch.device("cuda" if torch.cuda.is_available() and not C["force_cpu"] else "cpu")
amp = dev.type == "cuda"
log(f"device {dev} | torch {torch.__version__} | run cfg {json.dumps(C)}")
USE_KD, USE_FEAT, USE_CLIN = C["beta"] > 0, C["delta"] > 0, C["gamma"] > 0
USE_PROMPT = bool(C.get("diag_prompt", False))
assert not USE_PROMPT or USE_CLIN, "diagnosis prompts need the clinical head (gamma > 0)"

# ---------------------------------------------------------------- data ---------------------------------
D = torch.load(G["data_file"], weights_only=False)
imgs = np.load(G["npy"], mmap_mode="r")
L = G["max_len"]
LABELS = D["labels"]

TK = np.load(f"{G['teacher_dir']}/teacher_topk.npz")
tk_row = {int(v): k for k, v in enumerate(TK["idx"])}
tk_off, tk_ids, tk_lp = TK["offsets"], TK["ids"], TK["logp"]
K = tk_ids.shape[1]
TF = np.load(f"{G['teacher_dir']}/teacher_feats.npz")
tf_row = {int(v): k for k, v in enumerate(TF["idx"])}
FEATS = {"t_grid": TF["siglip_grid4"], "t_mean": TF["siglip_mean"], "t_proj": TF["proj_mean"]}
# per-dimension standardisation with TRAIN statistics: removes the large shared component of SigLIP features,
# so the cosine loss has to match image-specific directions instead of the common mean direction
tr_rows = np.array([tf_row[i] for i in D["train"]["idx"]])
FSTAT = {}
for k, a in FEATS.items():
    x = a[tr_rows].astype(np.float32).reshape(-1, a.shape[-1])
    FSTAT[k] = (x.mean(0), x.std(0) + 1e-3)
topk_ok = D["topk_ok"]                                   # idx -> bool, from teacher_cache.jsonl
STYLE = torch.tensor(D["style_mask"], dtype=torch.bool) if "style_mask" in D else None   # [VOCAB] True = formatting token

# ---- alignment check: teacher rows must equal the student's plain-text token count for every image
mis = {}
for split in ("train", "val"):
    bad = 0
    for i, ids in zip(D[split]["idx"], D[split]["ids"]):
        r = tk_row.get(i)
        n_rows = int(tk_off[r + 1] - tk_off[r]) if r is not None else 0
        if topk_ok.get(i, False) and n_rows != len(ids):
            bad += 1
    mis[split] = bad
n_ok = sum(1 for i in D["train"]["idx"] if topk_ok.get(i, False))
log(f"top-k alignment: rows != student tokens in train {mis['train']}, val {mis['val']} | "
    f"train images with teacher top-k {n_ok}/{len(D['train']['idx'])} | K={K}")
assert mis["train"] <= 0.005 * len(D["train"]["idx"]), "teacher top-k rows do not line up with the student tokens"


def encode_ids(ids):
    """input = [BOS] + t[:L-1]; labels = t[:L-1] + [EOS or the next real token if truncated]."""
    t = ids[: L - 1]
    nxt = ids[L - 1] if len(ids) > L - 1 else EOS_ID
    return [EOS_ID] + t, t + [nxt]


class DS(torch.utils.data.Dataset):
    def __init__(self, split, train):
        S = D[split]
        self.split, self.train = split, train
        self.idx, self.ids = S["idx"], S["ids"]
        self.cy, self.cm = S.get("clin_y"), S.get("clin_m")
        # sequence-level KD: per study, use the cleaned teacher impression instead of GT (decided per epoch by the trainer)
        self.ids_t = S.get("ids_t") if train else None
        self.use_t = np.zeros(len(self.idx), dtype=bool)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        gi = self.idx[i]
        inp, lab = encode_ids(self.ids_t[i] if self.use_t[i] else self.ids[i])
        item = {"img": torch.from_numpy(np.array(imgs[gi])), "inp": inp, "lab": lab, "pos": i}
        if not self.train:
            return item
        if USE_KD:
            r = tk_row.get(gi)
            n = len(self.ids[i])
            ok = topk_ok.get(gi, False) and r is not None and int(tk_off[r + 1] - tk_off[r]) == n
            m = min(n, len(lab)) if ok else 0          # label positions 0..m-1 have a teacher row; EOS has none
            a = int(tk_off[r]) if ok else 0
            item["kd_ids"], item["kd_lp"] = tk_ids[a:a + m], tk_lp[a:a + m]
        if USE_FEAT:
            r = tf_row[gi]
            for k, arr in FEATS.items():
                mu, sd = FSTAT[k]
                item[k] = torch.from_numpy((arr[r].astype(np.float32) - mu) / sd)
        if USE_CLIN:
            item["clin_y"], item["clin_m"] = torch.from_numpy(self.cy[i]), torch.from_numpy(self.cm[i])
        return item


def collate(b):
    n = max(len(x["inp"]) for x in b)
    B = len(b)
    out = {"img": torch.stack([x["img"] for x in b]), "pos": torch.tensor([x["pos"] for x in b])}
    inp = torch.full((B, n), EOS_ID, dtype=torch.long)
    lab = torch.full((B, n), -100, dtype=torch.long)
    att = torch.zeros((B, n), dtype=torch.long)
    for j, x in enumerate(b):
        inp[j, : len(x["inp"])] = torch.tensor(x["inp"]); lab[j, : len(x["lab"])] = torch.tensor(x["lab"])
        att[j, : len(x["inp"])] = 1
    out.update(inp=inp, att=att, labels=lab)
    if "kd_ids" in b[0]:
        ki = torch.zeros((B, n, K), dtype=torch.long)
        kl = torch.zeros((B, n, K), dtype=torch.float32)
        km = torch.zeros((B, n), dtype=torch.bool)
        for j, x in enumerate(b):
            m = len(x["kd_ids"])
            if m:
                ki[j, :m] = torch.from_numpy(x["kd_ids"].astype(np.int64))
                kl[j, :m] = torch.from_numpy(x["kd_lp"].astype(np.float32))
                km[j, :m] = True
        out.update(kd_ids=ki, kd_logp=kl, kd_mask=km)
    for k in ("t_grid", "t_mean", "t_proj", "clin_y", "clin_m"):
        if k in b[0]:
            out[k] = torch.stack([x[k] for x in b]).float()
    return out


TRAIN_GEN = torch.Generator(); TRAIN_GEN.manual_seed(C["seed"])   # one shuffle stream across per-epoch loader rebuilds
TARGET = C.get("target", "gt")                                    # "gt" | "s1" (teacher text, then GT) | "mix" (per-example coin)
assert TARGET in ("gt", "s1", "mix"), TARGET
assert TARGET == "gt" or not USE_KD, "token KD rows follow the GT tokens; do not combine with teacher-text targets"


def teacher_mask(ep):
    """bool [n_train]: which train studies use the teacher-text target in epoch ep."""
    n = len(D["train"]["idx"])
    if TARGET == "gt" or "ids_t" not in D["train"]:
        return np.zeros(n, dtype=bool)
    ok = np.asarray(D["train"]["teacher_ok"], dtype=bool)
    if TARGET == "s1":
        return ok & (ep < C["gt_from_epoch"])
    return ok & (np.random.RandomState(1000 * C["seed"] + ep).rand(n) < C["p_teacher"])


def loader(split, train, bs, use_t=None):
    g = TRAIN_GEN if train else torch.Generator().manual_seed(C["seed"])
    ds = DS(split, train)
    if use_t is not None:
        ds.use_t = use_t
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=train, collate_fn=collate, generator=g,
                                       num_workers=C["num_workers"], pin_memory=amp, drop_last=train,
                                       persistent_workers=C["num_workers"] > 0)


# ---------------------------------------------------------------- model --------------------------------
cfg = StudentConfig(mode=C["connector"], encoder_name=C["encoder"], img_size=C["img"], max_len=L,
                    pretrained_enc=C["pretrained"], pretrained_dec=C["pretrained"], diag_prompt=USE_PROMPT)
model = XRayReportStudent(cfg).to(dev)
MEAN = torch.tensor(model.enc_cfg["mean"], device=dev).view(1, 3, 1, 1)
STD = torch.tensor(model.enc_cfg["std"], device=dev).view(1, 3, 1, 1)
log(f"connector {C['connector']} | encoder {C['encoder']} @ {C['img']} | norm mean {model.enc_cfg['mean']} std {model.enc_cfg['std']}")
log("size", {k: f"{v / 1e6:.2f}M" for k, v in model.size_report().items()})

pw = None
if USE_CLIN:
    y, m = D["train"]["clin_y"], D["train"]["clin_m"]
    pos = (y * m).sum(0); neg = ((1 - y) * m).sum(0)
    pw = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1.0, C["pos_weight_max"]), dtype=torch.float32)
    log("clinical pos_weight", dict(zip(LABELS, np.round(pw.numpy(), 1).tolist())))
crit = DistillLoss(alpha=C["alpha"], beta=C["beta"], gamma=C["gamma"], delta=C["delta"], tau=C["tau"], pos_weight=pw,
                    kd_residual=C.get("kd_residual", False), kd_masked=C.get("kd_masked", False), style_mask=STYLE)


if USE_KD and C.get("kd_masked", False):
    from losses import kd_keep_mask
    ds0 = DS("train", True)
    nk = nt = 0
    for i in range(0, len(ds0), max(1, len(ds0) // 1000)):
        it = ds0[i]
        m = len(it["kd_ids"])
        if m:
            lab = torch.tensor(it["lab"][:m])
            km = kd_keep_mask(torch.from_numpy(it["kd_ids"].astype(np.int64))[None], torch.from_numpy(it["kd_lp"].astype(np.float32))[None],
                              lab[None], torch.ones(1, m, dtype=torch.bool), STYLE)
            nk += int(km.sum()); nt += m
            if it["pos"] == 0 or i == 0:
                log(f"masked-KD first-token check (study {i}): kept {bool(km[0, 0])} (expected False when the teacher top-1 is a formatting token)")
    log(f"masked KD: keeps {nk}/{nt} = {nk / max(nt, 1):.1%} of the label positions that have a teacher row (sample of ~1000 train studies); "
        f"formatting-token ids {int(STYLE.sum())}")
    assert nt > 0 and 0.3 < nk / nt < 0.99, "masked-KD kept fraction is implausible - check style_mask"


def prep(img_u8, train):
    """uint8 [B,512,512] -> normalised [B,3,IMG,IMG]; light geometric aug for training (no flips: laterality)."""
    x = img_u8.to(dev, non_blocking=True).float().div_(255).unsqueeze(1)
    x = F.interpolate(x, size=(C["img"], C["img"]), mode="bilinear", antialias=True, align_corners=False)
    if train and C["aug"]:
        B = x.size(0)
        ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(C["aug_rot_deg"])
        sc = 1 + (torch.rand(B, device=dev) * 2 - 1) * C["aug_scale"]
        tx, ty = [(torch.rand(B, device=dev) * 2 - 1) * C["aug_shift"] for _ in range(2)]
        th = torch.stack([torch.stack([torch.cos(ang) / sc, -torch.sin(ang) / sc, tx], 1),
                          torch.stack([torch.sin(ang) / sc, torch.cos(ang) / sc, ty], 1)], 1)
        grid = F.affine_grid(th, x.shape, align_corners=False)
        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    return (x.expand(-1, 3, -1, -1) - MEAN) / STD


def to_dev(b):
    return {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}


# ---------------------------------------------------------------- CheXbert for epoch selection ---------
if G.get("offline_test"):
    class _FakeCXB:                                       # local test only: no HF access
        def classes(self, texts, bs=64):
            return np.array([[(hash(t) >> k) % 4 for k in range(14)] for t in texts])

        @staticmethod
        def binary(cls, labels, convention="pos_only"):
            return (cls[:, : len(labels)] == 1).astype(np.int8)
    cxb = _FakeCXB()

    class _FakeTok:
        def decode(self, s, skip_special_tokens=True):
            return " ".join(f"w{t % 40}" for t in s)
    tok = _FakeTok()
else:
    from chexbert_labeler import CheXbert
    from transformers import AutoTokenizer
    cxb = CheXbert(device=str(dev))
    tok = AutoTokenizer.from_pretrained("distilgpt2")
binary = cxb.binary if G.get("offline_test") else CheXbert.binary


def micro_f1(P, Gt):
    P, Gt = P.astype(bool), Gt.astype(bool)
    tp = (P & Gt).sum(); fp = (P & ~Gt).sum(); fn = (~P & Gt).sum()
    return float(2 * tp / max(2 * tp + fp + fn, 1))


def f1b(y, p):
    tp = (y & p).sum()
    return float(2 * tp / max(2 * tp + (p & ~y).sum() + (~p & y).sum(), 1))


@torch.no_grad()
def set_prompt_thresholds():
    """A10: per-label head threshold = max F1 on VAL vs CheXbert(GT impression); same grid as the notebook's head scoring."""
    model.eval()
    P = []
    for b in loader("val", False, C["eval_batch"]):
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            P.append(model.clin_probs(model.encode(prep(b["img"], False))).float().cpu().numpy())
    model.train()
    P, Y = np.concatenate(P), D["val"]["gt_chexbert_pos"].astype(bool)
    thr = []
    for j in range(P.shape[1]):
        grid = np.unique(np.quantile(P[:, j], np.linspace(0.5, 0.995, 60)))
        thr.append(float(max(grid, key=lambda t: f1b(Y[:, j], P[:, j] >= t))) if Y[:, j].any() else 0.5)
    model.prompt_thr.copy_(torch.tensor(thr, dtype=torch.float32, device=dev))
    calls = (P >= np.array(thr)).sum(1)
    log(f"prompt thresholds (val max-F1): {dict(zip(LABELS, np.round(thr, 3).tolist()))} | positive prompts per val study "
        f"{calls.mean():.2f} (CheXbert GT {Y.sum(1).mean():.2f}, both incl. No Finding)")


@torch.no_grad()
def generate(split, bs, max_new, prompt_mode="head"):
    """prompt_mode (A10 only): "head" = head predictions at the VAL thresholds (the real model); "oracle" = CheXbert(GT impression)
    labels; "neg" = every label not positive. The last two are diagnostics on what the decoder does with the prompt."""
    model.eval()
    rows, clin, prom = [], [], []
    for b in loader(split, False, bs):
        x = prep(b["img"], False)
        st = None
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            pr = model.clin_probs(model.encode(x)) if USE_CLIN else None
            if USE_PROMPT:
                if prompt_mode == "oracle":
                    st = torch.as_tensor(np.asarray(D[split]["gt_chexbert_pos"])[b["pos"].numpy()], device=dev).long()
                elif prompt_mode == "neg":
                    st = torch.zeros(x.size(0), len(LABELS), dtype=torch.long, device=dev)
                else:
                    st = (pr >= model.prompt_thr).long()
                seqs, capped = model.greedy(x, max_new_tokens=max_new, prompt_states=st)
            else:
                seqs, capped = model.greedy(x, max_new_tokens=max_new)
        if pr is not None:
            clin.append(pr.float().cpu().numpy())
        if st is not None:
            prom.extend(";".join(c for c, v in zip(LABELS, r) if v) for r in st.tolist())
        for j, s, cp in zip(b["pos"].tolist(), seqs, capped):
            rows.append({"idx": int(D[split]["idx"][j]), "gt": D[split]["text"][j],
                         "pred": tok.decode(s, skip_special_tokens=True).strip(), "n_tok": len(s), "hit_cap": cp})
    model.train()
    df = pd.DataFrame(rows)
    if clin:
        P = np.concatenate(clin)
        for k, c in enumerate(LABELS):
            df[f"clinhead_{c}"] = P[:, k].round(4)
    if prom:
        df["prompt"] = prom
    return df


def val_f1(df):
    P = binary(cxb.classes(df.pred.astype(str).tolist()), LABELS, "pos_only")
    return micro_f1(P, D["val"]["gt_chexbert_pos"])


@torch.no_grad()
def val_ce():
    model.eval()
    tot, n = 0.0, 0
    for b in loader("val", False, C["eval_batch"]):
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            lg = model(prep(b["img"], False), b["inp"].to(dev), b["att"].to(dev), with_heads=False)["logits"]
        lab = b["labels"].to(dev)
        tot += float(F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), lab.reshape(-1), ignore_index=-100, reduction="sum"))
        n += int((lab != -100).sum())
    model.train()
    return tot / max(n, 1)


# ---------------------------------------------------------------- optimiser / schedule -----------------
model.set_stage("full")
opt = torch.optim.AdamW(model.param_groups(C["lr_new"], C["lr_pre"], C["wd"]), betas=(0.9, 0.999), eps=1e-8)
base_lrs = [g["lr"] for g in opt.param_groups]
tr_dl = loader("train", True, C["batch"])
steps_per_epoch = len(tr_dl) if C["max_steps_per_epoch"] <= 0 else min(len(tr_dl), C["max_steps_per_epoch"])
total = steps_per_epoch * C["epochs"]
warm = min(C["warmup"], max(1, total // 10))


GT_FROM_STEP = (C["gt_from_epoch"] - 1) * steps_per_epoch if TARGET == "s1" else None
REWARM = min(C.get("rewarm", 100), max(1, (total - GT_FROM_STEP) // 4)) if GT_FROM_STEP is not None else 0


def lr_factor(step):
    if GT_FROM_STEP is not None and step >= GT_FROM_STEP:     # S1 GT fine-tune: re-warm to half the peak, then cosine to 0
        s = step - GT_FROM_STEP
        if s < REWARM:
            return 0.5 * (s + 1) / REWARM
        p = (s - REWARM) / max(1, total - GT_FROM_STEP - REWARM)
        return 0.25 * (1 + math.cos(math.pi * min(1.0, p)))
    if step < warm:
        return (step + 1) / warm
    p = (step - warm) / max(1, total - warm)
    return 0.5 * (1 + math.cos(math.pi * min(1.0, p)))


scaler = torch.amp.GradScaler("cuda", enabled=amp)
deadline = G["deadline"]
step, best, best_ep, best_state, stopped_early = 0, -1.0, -1, None, False
curve = open(f"{OUT}/curve.jsonl", "w", buffering=1)
v0 = val_ce()
log(f"epoch 0 (before training) val_CE {v0:.4f}")
curve.write(json.dumps({"epoch": 0, "step": 0, "val_ce": v0}) + "\n")
model.train()
t_train = time.time()
for ep in range(1, C["epochs"] + 1):
    stage = "align" if ep <= C["align_epochs"] else "full"
    model.set_stage(stage)
    ntr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if TARGET != "gt":
        _ut = teacher_mask(ep)
        del tr_dl
        tr_dl = loader("train", True, C["batch"], use_t=_ut)
        log(f"epoch {ep} targets: teacher text for {_ut.mean():.1%} of train studies ({TARGET})"
            + (f", lr factor at epoch start {lr_factor(step):.3f}" if TARGET == "s1" else ""))
    log(f"epoch {ep} stage={stage} trainable={ntr / 1e6:.2f}M steps={steps_per_epoch}")
    agg, nb, t_ep, t_win = {}, 0, time.time(), time.time()
    gagg, gcnt = {}, {}
    for it, b in enumerate(tr_dl):
        if it >= steps_per_epoch:
            break
        for g, lr in zip(opt.param_groups, base_lrs):
            g["lr"] = lr * lr_factor(step)
        b = to_dev(b)
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            out = model(prep(b["img"], True), b["inp"], b["att"], with_heads=USE_FEAT or USE_CLIN,
                        **({"prompt_gt": b["clin_y"] * b["clin_m"], "p_gt": C["p_gt"]} if USE_PROMPT else {}))
        loss, parts = crit(out, b)                       # loss terms are computed in fp32 inside DistillLoss
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss at step {step}: {parts}")
        opt.zero_grad(set_to_none=True)
        if C.get("grad_probe_every", 0) and step % C["grad_probe_every"] == 0 and len(crit.terms) > 1:
            # per-term gradient norm over all trainable parameters (GradNorm-style diagnostic, Chen et al. 2018):
            # shows whether one weighted term dominates the update. Uses the AMP scale so fp16 gradients do not underflow.
            sc = float(scaler.get_scale()) if amp else 1.0
            tp = [p for p in model.parameters() if p.requires_grad]
            gp = {}
            for k, t in crit.terms.items():
                gs = torch.autograd.grad(t * sc, tp, retain_graph=True, allow_unused=True)
                v = math.sqrt(sum(float(g.float().pow(2).sum()) for g in gs if g is not None)) / sc
                if math.isfinite(v):
                    gp[k] = v
            for k, v in gp.items():
                gagg[k] = gagg.get(k, 0.0) + v; gcnt[k] = gcnt.get(k, 0) + 1
            log(f"  gradprobe step {step} " + " ".join(f"{k} {v:.3g}" for k, v in gp.items()))
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], C["clip"])
        scaler.step(opt); scaler.update()
        for k, v in parts.items():
            agg[k] = agg.get(k, 0.0) + v
        nb += 1; step += 1
        if step == C["timing_steps"]:
            dt = (time.time() - t_ep) / C["timing_steps"]
            mem = torch.cuda.max_memory_allocated() / 2**30 if amp else 0
            log(f"TIMING {dt:.3f} s/step (batch {C['batch']}, first {C['timing_steps']} steps) -> ~{dt * steps_per_epoch / 60:.1f} "
                f"min/epoch train only, ~{dt * total / 3600:.2f} h for {C['epochs']} epochs | peak GPU mem {mem:.2f} GiB")
        if step % C["log_every"] == 0:
            log(f"  step {step}/{total} " + " ".join(f"{k} {v / nb:.4f}" for k, v in agg.items()) +
                f" | lr_new {opt.param_groups[0]['lr']:.2e} grad_norm {float(gn):.2f} {(time.time() - t_win) / C['log_every']:.3f}s/step")
            t_win = time.time()
        if time.time() > deadline:
            stopped_early = True
            log("deadline reached: stopping training, going to evaluation")
            break
    t_ep_train = time.time() - t_ep
    if USE_PROMPT:
        set_prompt_thresholds()                        # before val CE / val decoding, which use the head's prompts
    v = val_ce()
    rec = {"epoch": ep, "step": step, "stage": stage, "val_ce": v, "train_min": round(t_ep_train / 60, 2),
           **{f"train_{k}": a / max(nb, 1) for k, a in agg.items()},
           **{f"gradnorm_{k}": gagg[k] / gcnt[k] for k in gagg}}
    if ep >= C["eval_from"] or ep == C["epochs"] or stopped_early:
        t_g = time.time()
        gdf = generate("val", C["gen_batch"], C["gen_max"])
        f1 = val_f1(gdf)
        rec.update(val_ce14_f1=f1, val_distinct=int(gdf.pred.nunique()), val_mean_tok=float(gdf.n_tok.mean()),
                   val_hit_cap=float(gdf.hit_cap.mean()), gen_min=round((time.time() - t_g) / 60, 2))
        if f1 > best:
            best, best_ep = f1, ep
            best_state = {k: t.detach().to("cpu", copy=True) for k, t in model.state_dict().items()}
    log(f"epoch {ep} done: train {t_ep_train / 60:.1f} min | " + " ".join(
        f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k not in ("epoch", "step", "stage")))
    curve.write(json.dumps(rec) + "\n")
    if stopped_early:
        break
train_minutes = (time.time() - t_train) / 60

# ---------------------------------------------------------------- final predictions with the best epoch -
log(f"best epoch {best_ep} (val CheXbert-14 micro-F1 {best:.4f})")
model.load_state_dict(best_state)
if C["save_weights"]:
    torch.save({"state_dict": {k: (t.half() if t.is_floating_point() else t) for k, t in best_state.items()},
                "student_config": cfg.__dict__, "run": RUN, "run_cfg": C, "best_epoch": best_ep}, f"{OUT}/best.pt")
del best_state
summary = {"run": RUN, "best_epoch": best_ep, "best_val_ce14_f1": best, "steps": step, "stopped_early": stopped_early,
           "train_minutes": round(train_minutes, 1), "params": model.size_report(), "cfg": C,
           "topk_misaligned": mis}
for split in ("val", "test"):
    t_g = time.time()
    df = generate(split, C["gen_batch"], C["gen_max"])
    df.to_csv(f"{OUT}/pred_{split}.csv", index=False)
    summary[f"{split}_gen_minutes"] = round((time.time() - t_g) / 60, 2)
    summary[f"{split}_mean_tok"] = float(df.n_tok.mean()); summary[f"{split}_hit_cap"] = float(df.hit_cap.mean())
    summary[f"{split}_distinct"] = int(df.pred.nunique())
    log(f"{split}: {len(df)} impressions | mean tok {df.n_tok.mean():.1f} | hit cap {df.hit_cap.mean():.1%} | distinct {df.pred.nunique()}")
for r in df.head(3).itertuples():
    log(f"  GT  : {r.gt[:160]!r}\n  PRED: {r.pred[:160]!r}" + (f"\n  PROMPT: {r.prompt!r}" if USE_PROMPT else ""))
if USE_PROMPT:
    summary["prompt_thr"] = dict(zip(LABELS, model.prompt_thr.tolist()))
    for mode in ("oracle", "neg"):
        t_g = time.time()
        dfx = generate("test", C["gen_batch"], C["gen_max"], prompt_mode=mode)
        dfx.to_csv(f"{OUT}/pred_test_{mode}.csv", index=False)
        summary[f"test_{mode}_gen_minutes"] = round((time.time() - t_g) / 60, 2)
        log(f"test ({mode} prompts): hit cap {dfx.hit_cap.mean():.1%} | distinct {dfx.pred.nunique()} | e.g. {dfx.pred.iloc[0][:120]!r}")
summary["minutes_total"] = round((time.time() - T0) / 60, 1)
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
log("DONE")
'''

CHEXBERT_LABELER_PY = r'''"""CheXbert labeler (Smit et al. 2020) for scoring generated report text.

Same architecture and checkpoint as the `f1chexbert` package (StanfordAIMI/RRG_scorers/chexbert.pth) but:
  * strict weight loading (only the known `position_ids` buffer may differ between transformers versions),
  * batched inference,
  * returns the raw 4-way classes so both label conventions can be computed:
      head classes 0 blank, 1 positive, 2 negative, 3 uncertain  (No Finding head: 0 blank, 1 positive)
      "pos_only"  : positive = class 1; uncertain counts as negative  (matches the IE643 baselines)
      "rrg"       : positive = class 1 or 3 (uncertain counts as positive; f1chexbert / RRG-paper convention)
"""
import numpy as np
import torch
import torch.nn as nn

CHEXBERT_ORDER = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema",
                  "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other",
                  "Fracture", "Support Devices", "No Finding"]


class _Labeler(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        h = bert.config.hidden_size
        self.linear_heads = nn.ModuleList([nn.Linear(h, 4) for _ in range(13)] + [nn.Linear(h, 2)])

    def forward(self, ids, att):
        cls = self.bert(ids, attention_mask=att)[0][:, 0]
        return [hd(cls) for hd in self.linear_heads]


class CheXbert:
    def __init__(self, device="cpu", checkpoint=None, random_init=False):
        from transformers import AutoConfig, AutoModel, BertTokenizer
        self.device = torch.device(device)
        self.tok = BertTokenizer.from_pretrained("bert-base-uncased")
        self.model = _Labeler(AutoModel.from_config(AutoConfig.from_pretrained("bert-base-uncased")))
        if not random_init:
            if checkpoint is None:
                from huggingface_hub import hf_hub_download
                checkpoint = hf_hub_download("StanfordAIMI/RRG_scorers", "chexbert.pth")
            sd = torch.load(checkpoint, map_location="cpu", weights_only=False)
            sd = sd.get("model_state_dict", sd)
            sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
            res = self.model.load_state_dict(sd, strict=False)
            bad_missing = [k for k in res.missing_keys if not k.endswith("position_ids")]
            bad_unexp = [k for k in res.unexpected_keys if not k.endswith("position_ids")]
            assert not bad_missing and not bad_unexp, f"CheXbert weights mismatch: missing {bad_missing[:5]} unexpected {bad_unexp[:5]}"
        self.model.to(self.device).eval()

    @staticmethod
    def _clean(t):
        return " ".join(str(t).strip().replace("\n", " ").split())

    @torch.no_grad()
    def classes(self, texts, bs=64):
        """-> int array [N,14] of raw classes in CHEXBERT_ORDER."""
        out = []
        for s in range(0, len(texts), bs):
            chunk = [self._clean(t) for t in texts[s:s + bs]]
            # [CLS] + wordpieces + [SEP], truncated to 512 (same ids as the reference tokenize -> encode_plus path;
            # encode_plus was removed in transformers 5)
            enc = [self.tok(t, add_special_tokens=True, truncation=True, max_length=512)["input_ids"] if t else
                   [self.tok.cls_token_id, self.tok.sep_token_id] for t in chunk]
            n = max(len(e) for e in enc)
            ids = torch.zeros(len(enc), n, dtype=torch.long)
            att = torch.zeros(len(enc), n, dtype=torch.float)
            for i, e in enumerate(enc):
                ids[i, :len(e)] = torch.tensor(e); att[i, :len(e)] = 1
            logits = self.model(ids.to(self.device), att.to(self.device))
            out.append(torch.stack([l.argmax(1) for l in logits], 1).cpu())
        return torch.cat(out).numpy()

    @staticmethod
    def binary(cls, labels, convention="pos_only"):
        """raw classes [N,14] -> 0/1 matrix [N,len(labels)] in the order of `labels`."""
        J = [CHEXBERT_ORDER.index(c) for c in labels]
        c = cls[:, J]
        pos = (c == 1) if convention == "pos_only" else ((c == 1) | (c == 3))
        return pos.astype(np.int8)
'''
print('modules embedded:', {k: len(v) for k, v in [('model', MODEL_PY), ('losses', LOSSES_PY), ('worker', DISTILL_WORKER_PY), ('chexbert', CHEXBERT_LABELER_PY)]})

In [ ]:
import os, sys, json, glob, re, subprocess, shutil, threading, zlib
import numpy as np, pandas as pd

if not OFFLINE_TEST:
    try:
        import pycocoevalcap
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap"], check=True)
import torch, timm, transformers
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | timm {timm.__version__} | transformers {transformers.__version__} "
      f"| java: {shutil.which('java') or 'NOT FOUND (METEOR skipped)'}")

SRC_DIR = f"{WORK}/distill_src"
os.makedirs(SRC_DIR, exist_ok=True)
for n, s in {"model.py": MODEL_PY, "losses.py": LOSSES_PY, "distill_worker.py": DISTILL_WORKER_PY, "chexbert_labeler.py": CHEXBERT_LABELER_PY}.items():
    open(f"{SRC_DIR}/{n}", "w").write(s)
sys.path.insert(0, SRC_DIR)

KNOWN_CSV = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning-step-7/reports_clean.csv"
KNOWN_NPY = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning/images_512.npy"

def resolve_data():
    csv, npy, tdir = CSV_PATH, NPY_PATH, TEACHER_DIR
    if csv is None and os.path.exists(KNOWN_CSV):
        csv = KNOWN_CSV
    if npy is None and os.path.exists(KNOWN_NPY):
        npy = KNOWN_NPY
    if csv is None:
        c = [p for p in glob.glob(f"{INPUT_ROOT}/**/reports_clean.csv", recursive=True)
             if {"impression_final", "use_for_training"} <= set(pd.read_csv(p, nrows=0).columns)]
        assert len(c) == 1, f"Need exactly one reports_clean.csv containing impression_final; found {c}. Attach 'IE643 Cleaning Step 7'."
        csv = c[0]
    if npy is None:
        c = glob.glob(f"{INPUT_ROOT}/**/images_512.npy", recursive=True)
        assert len(c) == 1, f"Need exactly one images_512.npy; found {c}. Attach 'IE643 Cleaning'."
        npy = c[0]
    if tdir is None:
        c = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT_ROOT}/**/teacher_final/merge_report.json", recursive=True)})
        assert len(c) == 1, f"Need exactly one teacher_final/ folder; found {c}. Attach the merge version of the teacher-cache notebook."
        tdir = c[0]
    for p in (csv, npy, f"{tdir}/teacher_topk.npz", f"{tdir}/teacher_feats.npz", f"{tdir}/teacher_cache.jsonl"):
        assert os.path.exists(p), f"missing {p}"
    return csv, npy, tdir

CSV, NPY, TDIR = resolve_data()
df = pd.read_csv(CSV, low_memory=False)
imgs = np.load(NPY, mmap_mode="r")
assert len(df) == imgs.shape[0] and (df["idx"].values == np.arange(len(df))).all(), "CSV and images_512.npy are not aligned"
df["impression_final"] = df["impression_final"].fillna("").astype(str)
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
U = df[df.use_for_training]
MR = json.load(open(f"{TDIR}/merge_report.json"))
assert MR["missing_from_SPLITS"] == 0 and MR["features_finite"], MR
TC = {}
for line in open(f"{TDIR}/teacher_cache.jsonl"):
    r = json.loads(line)
    TC[r["idx"]] = r
print(f"csv: {CSV}\nnpy: {NPY}\nteacher: {TDIR} ({MR['cached']} images, by split {MR['by_split']})")
print(f"rows {len(df)} | usable per split {U.subset.value_counts().to_dict()} | teacher top-k ok: "
      f"{sum(r.get('topk_ok', False) for r in TC.values())}/{len(TC)}")

LABELS = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
          "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"]
FIVE = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
assert all(c in df.columns for c in LABELS), [c for c in LABELS if c not in df.columns]

# ---- runs finished in earlier sessions (this notebook's own output attached as input) are imported, not re-run
RUN_DIR = f"{WORK}/distill_runs" + ("_smoke" if MODE == "smoke" else "")   # smoke runs never land where real runs are imported from
os.makedirs(RUN_DIR, exist_ok=True)
for s in ([] if MODE == "smoke" else sorted(glob.glob(f"{INPUT_ROOT}/**/distill_runs/*/summary.json", recursive=True))):
    d = os.path.dirname(s); name = os.path.basename(d)
    _c = json.load(open(s)).get("cfg", {})
    if _c.get("force_cpu") or _c.get("max_steps_per_epoch", 0) > 0:
        print(f"NOT imported (smoke/CPU run, not a real result): {s}")
        continue
    if not os.path.exists(f"{RUN_DIR}/{name}/summary.json"):
        shutil.copytree(d, f"{RUN_DIR}/{name}", dirs_exist_ok=True)
        print("imported finished run from an earlier session:", name)

In [ ]:
# ---- unit checks of the loss (CPU, < 1 s): catches a broken KD/feature/clinical term before any GPU time is used
from losses import topk_kd_loss, feat_loss, clin_loss, topk_kd_masked_loss, kd_keep_mask
g = torch.Generator().manual_seed(0)
V, Kk = 50257, 20
sup = torch.stack([torch.randperm(V, generator=g)[:Kk] for _ in range(8)]).view(2, 4, Kk)
tl = torch.randn(2, 4, Kk, generator=g)
logits = torch.full((2, 4, V), -100.0)
for i in range(2):
    for j in range(4):
        logits[i, j, sup[i, j]] = tl[i, j]
tlogp, mask = torch.log_softmax(tl, -1), torch.ones(2, 4, dtype=torch.bool)
k0 = float(topk_kd_loss(logits, sup, tlogp, mask, tau=2.0))
k1 = float(topk_kd_loss(logits, torch.cat([sup, torch.full((2, 4, 1), 50295)], -1), torch.cat([tlogp, torch.full((2, 4, 1), -0.01)], -1), mask, tau=2.0))
k2 = float(topk_kd_loss(torch.randn(2, 4, V, generator=g), sup, tlogp, mask, tau=2.0))
k3 = float(topk_kd_loss(torch.randn(2, 4, V, generator=g), sup, tlogp, torch.zeros_like(mask), tau=2.0))
x = torch.randn(3, 16, 8, generator=g)
f0, f1 = float(feat_loss(x, 2 * x)), float(feat_loss(x, -x))
c0 = float(clin_loss(torch.tensor([[10.0, -10.0]]), torch.tensor([[1.0, 1.0]]), torch.tensor([[1.0, 0.0]])))
print(f"KD student==teacher {k0:.2e} | + teacher-only id {k1:.2e} | random student {k2:.2f} | empty mask {k3} | "
      f"feat same-dir {f0:.1e} opposite {f1:.2f} | clin masked {c0:.1e}")
assert k0 < 1e-4 and abs(k1 - k0) < 1e-4 and k2 > 1 and k3 == 0 and f0 < 1e-5 and abs(f1 - 2) < 1e-5 and c0 < 1e-3
# masked KD: 4 positions in one study. p0: top-1 is a formatting token -> dropped; p1: GT not in top-k -> dropped;
# p2: usable, but a formatting token sits in the top-k -> kept, that token removed from the target; p3: top-1 is EOS -> dropped.
Vt, Kt = 60, 4
sty = torch.zeros(Vt, dtype=torch.bool); sty[[7, 8]] = True
ids_ = torch.tensor([[[7, 1, 2, 3], [1, 2, 3, 4], [1, 8, 2, 3], [Vt - 1, 1, 2, 3]]])      # id Vt-1 plays the role of EOS here
lp_ = torch.log_softmax(torch.tensor([[[3., 2, 1, 0], [3., 2, 1, 0], [3., 2.5, 1, 0], [3., 2, 1, 0]]]), -1)
lab_ = torch.tensor([[1, 9, 1, 1]]); pm_ = torch.ones(1, 4, dtype=torch.bool)
keep_ = kd_keep_mask(ids_, lp_, lab_, pm_, sty, vocab=Vt, eos=Vt - 1)
assert keep_.tolist() == [[False, False, True, False]], keep_
# student that equals the cleaned (formatting token removed, renormalised) target at p2 -> loss 0
tgt = torch.softmax(torch.tensor([3., 1., 0.]), -1)                                     # ids 1, 2, 3 after dropping id 8
lg_ = torch.full((1, 4, Vt), -30.0); lg_[0, 2, [1, 2, 3]] = tgt.log()
mk, kept_ = topk_kd_masked_loss(lg_, ids_, lp_, lab_, pm_, sty, tau=1.0, vocab=Vt, eos=Vt - 1)
assert float(mk) < 1e-5 and abs(float(kept_) - 0.25) < 1e-6, (float(mk), float(kept_))
# a student putting its mass on the formatting token is NOT rewarded: raising logit 8 must not lower the loss
lg2 = lg_.clone(); lg2[0, 2, 8] = 5.0
mk2, _ = topk_kd_masked_loss(lg2, ids_, lp_, lab_, pm_, sty, tau=1.0, vocab=Vt, eos=Vt - 1)
assert float(mk2) > float(mk) + 0.1
print(f"masked KD: keep mask {keep_.tolist()[0]} | cleaned-target match {float(mk):.1e} | formatting-token mass penalised {float(mk2):.2f} | kept {float(kept_):.2f}")
print("LOSS CHECKS OK")

In [ ]:
# ---- tokenise impressions (DistilGPT2 ids == teacher ids), clinical targets, CheXbert labels of GT + teacher text
if OFFLINE_TEST:
    tok_encode = lambda t: [zlib.crc32(w.encode()) % 50000 for w in t.split()]
else:
    from transformers import AutoTokenizer
    _tok = AutoTokenizer.from_pretrained("distilgpt2")
    tok_encode = lambda t: _tok(t)["input_ids"]

SPL = {s: U[U.subset == s].reset_index(drop=True) for s in ("train", "val", "test")}
if MODE == "smoke":
    SPL = {s: d.sample(n, random_state=SEED).reset_index(drop=True) for (s, d), n in zip(SPL.items(), (48, 16, 16))}
ids = {s: [tok_encode(t) for t in d.impression_final] for s, d in SPL.items()}
lens = np.array([len(x) + 1 for x in ids["train"]])
if MAX_LEN is None:
    MAX_LEN = int(min(256, np.ceil(np.percentile(lens, 95))))
print(f"train impression tokens incl. EOS: median {np.median(lens):.0f}, p95 {np.percentile(lens, 95):.0f}, max {lens.max()} "
      f"-> MAX_LEN = {MAX_LEN}; truncated {(lens > MAX_LEN).mean():.1%}")

# clinical targets from the CSV labels: 1 = positive; uncertain (-1) is ignored (CheXpert "U-Ignore"); blank/0 = negative
def clin_targets(d):
    y = np.zeros((len(d), len(LABELS)), np.float32); m = np.ones_like(y)
    for j, c in enumerate(LABELS):
        v = pd.to_numeric(d["y_no_finding"] if c == "No Finding" else d[c], errors="coerce")
        y[:, j] = (v == 1); m[:, j] = (v != -1)
    return y, m
print("CSV label values (train):", {c: pd.to_numeric(SPL["train"][c], errors="coerce").value_counts(dropna=False).to_dict() for c in LABELS[1:4]}, "...")
ym = [c for c in df.columns if re.match(r"^[ym]_", c)]
print(f"y_/m_ columns in the CSV ({len(ym)}): {ym[:6]} ...  (shown for the record; the targets above use the raw label columns)")
for c in ("Cardiomegaly", "Pleural Effusion"):
    yc, mc = f"y_{c.lower().replace(' ', '_')}", f"m_{c.lower().replace(' ', '_')}"
    if yc in df.columns:
        print(pd.crosstab(pd.to_numeric(U[c], errors="coerce").fillna(-9), [U[yc]] + ([U[mc]] if mc in df.columns else []), dropna=False).to_string())

if OFFLINE_TEST:
    class _FakeCXB:
        def classes(self, texts, bs=64):
            return np.array([[(hash(t) >> k) % 4 for k in range(14)] for t in texts])
        @staticmethod
        def binary(cls, labels, convention="pos_only"):
            c = cls[:, :len(labels)]
            return ((c == 1) | ((c == 3) & (convention != "pos_only"))).astype(np.int8)
    cxb = _FakeCXB(); CXB = _FakeCXB
else:
    from chexbert_labeler import CheXbert
    from huggingface_hub import hf_hub_download
    hf_hub_download("StanfordAIMI/RRG_scorers", "chexbert.pth")       # download once; workers read the HF cache
    cxb = CheXbert(device="cuda" if torch.cuda.is_available() else "cpu"); CXB = CheXbert

GTCLS, TCHCLS = {}, {}
for s in ("val", "test"):
    GTCLS[s] = cxb.classes(SPL[s].impression_final.tolist())
    TCHCLS[s] = cxb.classes([TC[int(i)]["impression"] for i in SPL[s].idx])
# formatting-token mask for masked token KD (A9): same rule as ie643_kd_diagnostic - decoded token contains * [ ] # or a newline, or is a bare list number
STYLE_RE = re.compile(r"[\*\[\]#\n]|^\s*\d+\.?\s*$")
if OFFLINE_TEST:
    _dec = lambda i: f"w{i}"
    STYLE_MASK = np.array([i % 97 == 0 for i in range(50257)])
else:
    STYLE_MASK = np.array([bool(STYLE_RE.search(_tok.decode([i]))) for i in range(50257)])
print(f"formatting-token mask: {int(STYLE_MASK.sum())} of 50257 ids, e.g. {[ _tok.decode([i]) for i in np.where(STYLE_MASK)[0][:8]] if not OFFLINE_TEST else '(offline)'}")
if not OFFLINE_TEST:
    for _t, _exp in (("[", True), (" [", True), ("**", True), (" **", True), ("\n", True), (".", False), (" the", False), (",", False), (" no", False)):
        assert STYLE_MASK[_tok(_t)["input_ids"][0]] == _exp, (_t, _exp)
data = {"labels": LABELS, "topk_ok": {int(i): bool(r.get("topk_ok", False)) for i, r in TC.items()},
        "tokenisation": "distilgpt2, plain text (no leading space), no lower-casing", "style_mask": STYLE_MASK.tolist()}
# ---- sequence-level KD targets (v7): the teacher's own impression, cleaned to the GT style (lower case, no markdown / list syntax)
_CAPKEYS = sorted({k for r in TC.values() for k in r if re.search(r"imp.*cap|cap.*imp", k, re.I)})
print("teacher_cache.jsonl keys:", sorted(next(iter(TC.values())).keys()), "| impression cap-flag keys used:", _CAPKEYS)
def clean_teacher(t):
    t = str(t or "")
    t = re.sub(r"\[[^\]]{0,60}\]", " ", t)                       # bracket tags such as [1] or [Lungs]
    t = t.replace("**", " ").replace("#", " ").replace("*", " ")
    t = re.sub(r"(?:(?<=\s)|^)\d+[.)]\s+(?=[A-Za-z])", " ", t)       # list numbers "1. " / "2) " (not "4.5 cm")
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t
_tt = [clean_teacher(TC[int(i)]["impression"]) for i in SPL["train"].idx]
_capped = [any(bool(TC[int(i)].get(k)) for k in _CAPKEYS) for i in SPL["train"].idx]
TEACHER_OK = [bool(t) and len(t.split()) >= 2 and not c for t, c in zip(_tt, _capped)]
ids_t = [tok_encode(t) if ok else g for t, ok, g in zip(_tt, TEACHER_OK, ids["train"])]
_lt = np.array([len(x) + 1 for x, ok in zip(ids_t, TEACHER_OK) if ok])
_iv = lambda ts: float(np.mean([str(t).startswith("interval") for t in ts]))
print(f"teacher-text targets (train): usable {np.mean(TEACHER_OK):.1%} ({sum(TEACHER_OK)}/{len(TEACHER_OK)}; capped {sum(_capped)}) | "
      f"tokens incl. EOS median {np.median(_lt) if len(_lt) else 0:.0f}, p95 {np.percentile(_lt, 95) if len(_lt) else 0:.0f}, "
      f"> MAX_LEN {(_lt > MAX_LEN).mean() if len(_lt) else 0:.1%} | starts with 'interval': teacher {_iv(_tt):.1%} vs GT "
      f"{_iv(SPL['train'].impression_final):.1%} | distinct teacher {len(set(_tt))} vs GT {SPL['train'].impression_final.nunique()}")
for _j in range(min(3, len(_tt))):
    print(f"  raw    : {str(TC[int(SPL['train'].idx[_j])]['impression'])[:150]!r}\n  cleaned: {_tt[_j][:150]!r}")
assert np.mean(TEACHER_OK) > 0.9 or MODE == "smoke", "fewer than 90% usable teacher impressions - check the cleaner"
for s, d in SPL.items():
    y, m = clin_targets(d)
    data[s] = {"idx": d.idx.astype(int).tolist(), "ids": ids[s], "text": d.impression_final.tolist(), "clin_y": y, "clin_m": m}
    if s != "train":
        data[s]["gt_chexbert_pos"] = CXB.binary(GTCLS[s], LABELS, "pos_only")
data["train"].update(ids_t=ids_t, teacher_ok=TEACHER_OK, text_t=_tt)
# diagnostic for the feature targets: how much of each cached teacher vector is a component shared by all images?
# (high raw pairwise cosine -> a cosine loss on raw targets is mostly satisfied by predicting the common direction)
_TF = np.load(f"{TDIR}/teacher_feats.npz")
_rows = {int(v): k for k, v in enumerate(_TF["idx"])}
_sel = np.array([_rows[int(i)] for i in SPL["train"].idx[:500]])
FEAT_DIAG = {}
for _k in ("siglip_mean", "proj_mean", "siglip_grid4"):
    _x = _TF[_k][_sel].astype(np.float32).reshape(len(_sel), -1, _TF[_k].shape[-1]).mean(1)
    _z = (_x - _x.mean(0)) / (_x.std(0) + 1e-3)
    def _pc(a):
        a = a / np.linalg.norm(a, axis=1, keepdims=True); c = a @ a.T
        return float(c[~np.eye(len(a), dtype=bool)].mean())
    FEAT_DIAG[_k] = {"raw_mean_pairwise_cos": round(_pc(_x), 3), "standardised_mean_pairwise_cos": round(_pc(_z), 3)}
print("teacher feature targets (train subset):", FEAT_DIAG)
del _TF
DATA_FILE = f"{WORK}/distill_data.pt"
torch.save(data, DATA_FILE)
print(f"data file: {DATA_FILE} | " + " | ".join(f"{s} {len(d)}" for s, d in SPL.items()))
print("train positive rate:", dict(zip(LABELS, (data['train']['clin_y'] * data['train']['clin_m']).mean(0).round(3).tolist())))
del cxb
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ---- run configs
smoke = MODE == "smoke"
OTHER = "prefix" if CONNECTOR == "xattn" else "xattn"
FULL = dict(beta=BETA, delta=DELTA, gamma=GAMMA)
RUN_SETS = {
    # A1 = Hinton/DistilBERT soft targets (renormalised top-k, tau 2); A1r = residual-to-GT top-k at tau 1, one of the
    # corrections for the top-k bias evaluated by Anshumann et al. 2025. The better KD variant is then used in A2/A3-style runs.
    "ablation1": {"A0_ce": {}, "A1_ce_kd": dict(beta=BETA), "A1t1_ce_kd": dict(beta=BETA, tau=1.0), "A1r_ce_kdresid": dict(beta=BETA, tau=1.0, kd_residual=True),
                  "A2_ce_kd_feat": dict(beta=BETA, delta=DELTA), "A3_full": FULL},
    # A9: token KD with teacher targets cleaned per the KD diagnostic (see losses.kd_keep_mask); tau 1 (the less harmful setting in ablation 1)
    # order matters for the 2-GPU split todo[g::2]: GPU0 gets A6, A8; GPU1 gets A7, A9
    "ablation3": {"A6_ce_clin": dict(gamma=GAMMA), "A7_ce_feat": dict(delta=DELTA), "A8_ce_clin_feat": dict(gamma=GAMMA, delta=DELTA),
                  "A9_ce_kdmask": dict(beta=BETA, tau=1.0, kd_masked=True)},
    # final loss = CE only (ablation 3). Order matters for the 2-GPU split todo[g::2]: GPU0 gets A4, A5; GPU1 gets the two seed repeats.
    "ablation2": {f"A4_ce_{OTHER}": dict(connector=OTHER), "A0s1_ce": dict(seed=1), "A5_ce_224": dict(encoder=ENCODER_224, img=224), "A0s2_ce": dict(seed=2)},
    "ablation2_fullloss": {f"A4_full_{OTHER}": dict(FULL, connector=OTHER), "A5_full_224": dict(FULL, encoder=ENCODER_224, img=224)},
    # A10 diagnosis prompts (A6 + prompts), two seeds; order -> GPU0 gets A10, GPU1 gets A10s1
    "prompt": {"A10_ce_clin_prompt": dict(gamma=GAMMA, diag_prompt=True), "A10s1_ce_clin_prompt": dict(gamma=GAMMA, diag_prompt=True, seed=1)},
    # v7 sequence-level KD (CE only); order -> GPU0 gets seed 0, GPU1 gets seed 1
    "seqkd1": {"S1_seqkd_ft": dict(target="s1"), "S1s1_seqkd_ft": dict(target="s1", seed=1)},
    "seqkd2": {"S2_mix": dict(target="mix"), "S2s1_mix": dict(target="mix", seed=1)},
    "custom": CUSTOM_RUNS,
}
RUNS = {"A0_ce": {}, "A10_ce_clin_prompt": dict(gamma=GAMMA, diag_prompt=True),
        "S1_seqkd_ft": dict(target="s1", gt_from_epoch=2), "S1s1_seqkd_ft": dict(target="s1", gt_from_epoch=2, seed=1),
        "S2_mix": dict(target="mix"), "S2s1_mix": dict(target="mix", seed=1)} if smoke else RUN_SETS[RUN_SET]
DEFAULTS = dict(seed=SEED, connector=CONNECTOR, encoder=ENCODER_384, img=384, pretrained=PRETRAINED, force_cpu=smoke,
                epochs=2 if smoke else EPOCHS, align_epochs=1 if smoke else ALIGN_EPOCHS, eval_from=2 if smoke else EVAL_FROM,
                max_steps_per_epoch=3 if smoke else 0, batch=4 if smoke else BATCH, eval_batch=8 if smoke else 32,
                gen_batch=8 if smoke else 64, gen_max=24 if smoke else MAX_LEN,
                lr_new=LR_NEW, lr_pre=LR_PRE, wd=WD, warmup=WARMUP, clip=CLIP,
                aug=True, aug_rot_deg=5.0, aug_scale=0.05, aug_shift=0.03,
                alpha=1.0, beta=0.0, gamma=0.0, delta=0.0, tau=TAU, kd_residual=False, kd_masked=False, pos_weight_max=POS_WEIGHT_MAX, diag_prompt=False, p_gt=P_GT,
                target="gt", gt_from_epoch=SEQKD_GT_FROM, p_teacher=P_TEACHER, rewarm=100,
                num_workers=2, log_every=2 if smoke else 100, grad_probe_every=2 if smoke else 100, timing_steps=3 if smoke else 30, save_weights=SAVE_WEIGHTS and not smoke)
todo = [r for r in RUNS if not os.path.exists(f"{RUN_DIR}/{r}/summary.json")]
CFG = {"defaults": DEFAULTS, "runs": RUNS, "data_file": DATA_FILE, "npy": NPY, "teacher_dir": TDIR, "max_len": MAX_LEN,
       "deadline": T0 + (BUDGET_H - 0.35) * 3600, "offline_test": OFFLINE_TEST, "version": NOTEBOOK_VERSION}
json.dump(CFG, open(f"{RUN_DIR}/cfg_{RUN_SET if not smoke else 'smoke'}.json", "w"), indent=1)
CFG_PATH = f"{RUN_DIR}/cfg_{RUN_SET if not smoke else 'smoke'}.json"
for r, o in RUNS.items():
    print(f"  {r:16s} {'(already done)' if r not in todo else ''} {o}")

def n_gpus():
    try:
        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
    except FileNotFoundError:
        return 0
NG = 0 if smoke else n_gpus()
print(f"GPUs visible: {NG} | runs to do: {todo}")
if not smoke:
    assert NG >= 1, "train mode needs a GPU accelerator (T4 x2). Use MODE='smoke' on CPU."

PRINT_KEYS = ("TIMING", "epoch", "alignment", "best epoch", "val:", "test:", "DONE", "deadline", "Error", "error", "Traceback", "pos_weight", "masked", "prompt", "targets")
print_lock = threading.Lock()

def run_one(name, gpu):
    out = f"{RUN_DIR}/{name}"
    os.makedirs(out, exist_ok=True)
    env = dict(os.environ, TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1", CUDA_VISIBLE_DEVICES="" if gpu is None else str(gpu))
    with open(f"{out}/stdout.txt", "w") as f:
        p = subprocess.Popen([sys.executable, f"{SRC_DIR}/distill_worker.py", CFG_PATH, name, out], stdout=f, stderr=subprocess.STDOUT, env=env)
        seen = 0
        while True:
            alive = p.poll() is None
            lines = open(f"{out}/stdout.txt").read().splitlines()
            with print_lock:
                for ln in lines[seen:]:
                    if any(k in ln for k in PRINT_KEYS):
                        print(ln, flush=True)
            seen = len(lines)
            if not alive:
                break
            time.sleep(5 if smoke else 30)
    if p.returncode != 0:
        with print_lock:
            print(f"\n----- {name} FAILED (exit {p.returncode}); last 40 lines -----\n" + "\n".join(lines[-40:]))
    return p.returncode

STATUS = {}
def gpu_queue(names, gpu):
    est_h = EST_RUN_H
    for name in names:
        left_h = (T0 + BUDGET_H * 3600 - time.time()) / 3600
        if not smoke and left_h < est_h:
            STATUS[name] = f"not started ({left_h:.1f} h left < {est_h:.1f} h needed): run it in the next session"
            continue
        t = time.time()
        STATUS[name] = "ok" if run_one(name, gpu) == 0 else "FAILED"
        est_h = 1.1 * (time.time() - t) / 3600

t_all = time.time()
if smoke or NG == 1:
    gpu_queue(todo, None if smoke else 0)
else:
    th = [threading.Thread(target=gpu_queue, args=(todo[g::NG], g)) for g in range(NG)]
    [t.start() for t in th]; [t.join() for t in th]
print(f"\nall runs finished in {(time.time() - t_all) / 60:.1f} min | status {STATUS}")
assert "FAILED" not in STATUS.values(), "a run failed - see its log above"

In [ ]:
# ---- metrics (same text normalisation + NLG code as the baselines and pilot notebooks)
def text_tokens(t):
    return " ".join(re.findall(r"[a-z0-9]+", str(t).lower()))

def nlg_metrics(preds, refs):
    if OFFLINE_TEST:
        return {"BLEU-1": None, "BLEU-4": None, "ROUGE-L": None, "METEOR": None, "CIDEr": None}
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    g = {i: [text_tokens(r) or "empty"] for i, r in enumerate(refs)}
    p = {i: [text_tokens(x) or "empty"] for i, x in enumerate(preds)}
    out = {}
    try:
        b, _ = Bleu(4).compute_score(g, p, verbose=0)
    except TypeError:
        b, _ = Bleu(4).compute_score(g, p)
    out.update({f"BLEU-{k + 1}": float(v) for k, v in enumerate(b)})
    out["ROUGE-L"] = float(Rouge().compute_score(g, p)[0])
    out["CIDEr"] = float(Cider().compute_score(g, p)[0])
    try:
        from pycocoevalcap.meteor.meteor import Meteor
        out["METEOR"] = float(Meteor().compute_score(g, p)[0])
    except Exception as e:
        out["METEOR"] = None
        print("METEOR skipped (needs Java):", repr(e)[:200])
    return out

def label_matrix(rows):
    m = np.zeros((len(rows), len(LABELS)), dtype=np.int8)
    for j, c in enumerate(LABELS):
        m[:, j] = (pd.to_numeric(rows["y_no_finding"] if c == "No Finding" else rows[c], errors="coerce").values == 1)
    return m

def micro_f1(P, G, cols=LABELS):
    J = [LABELS.index(c) for c in cols]
    P, G = P[:, J].astype(bool), G[:, J].astype(bool)
    tp = (P & G).sum(); fp = (P & ~G).sum(); fn = (~P & G).sum()
    return float(2 * tp / max(2 * tp + fp + fn, 1))

def per_label_f1(P, G):
    return {c: micro_f1(P, G, [c]) for c in LABELS}

def boot_ci(P, G, cols, n=1000, seed=0):
    rs = np.random.RandomState(seed)
    v = [micro_f1(P[s], G[s], cols) for s in (rs.randint(0, len(P), len(P)) for _ in range(n))]
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]

def paired_boot(Pa, Pb, G, cols, n=2000, seed=0):
    rs = np.random.RandomState(seed); d = []
    for _ in range(n):
        s = rs.randint(0, len(G), len(G))
        d.append(micro_f1(Pb[s], G[s], cols) - micro_f1(Pa[s], G[s], cols))
    return float(np.mean(d)), [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]
print("metric functions ready")

In [ ]:
# ---- evaluation of every finished run (this session + imported) on val and test, plus the teacher as a reference
cxb = CXB(device="cuda" if torch.cuda.is_available() else "cpu") if not OFFLINE_TEST else _FakeCXB()
runs = sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob(f"{RUN_DIR}/*/summary.json"))
if smoke:
    runs = [r for r in runs if r in RUNS]
GT = {s: CXB.binary(GTCLS[s], LABELS, "pos_only") for s in ("val", "test")}
GTR = {s: CXB.binary(GTCLS[s], LABELS, "rrg") for s in ("val", "test")}
GCSV = {s: label_matrix(SPL[s]) for s in ("val", "test")}
ORDER = {s: SPL[s].idx.astype(int).tolist() for s in ("val", "test")}

def score(preds, s, P_cls=None):
    cls = cxb.classes(list(preds)) if P_cls is None else P_cls
    P, PR = CXB.binary(cls, LABELS, "pos_only"), CXB.binary(cls, LABELS, "rrg")
    r = nlg_metrics(list(preds), SPL[s].impression_final.tolist())
    r.update({"CE14_F1": micro_f1(P, GT[s]), "CE14_F1_95CI": boot_ci(P, GT[s], LABELS), "CE5_F1": micro_f1(P, GT[s], FIVE),
              "CE14_F1_vs_CSV": micro_f1(P, GCSV[s]), "CE5_F1_vs_CSV": micro_f1(P, GCSV[s], FIVE),
              "CE14_F1_rrg": micro_f1(PR, GTR[s]), "CE5_F1_rrg": micro_f1(PR, GTR[s], FIVE),
              "per_label_F1": per_label_f1(P, GT[s]), "distinct": int(pd.Series(list(preds)).nunique())})
    # template-collapse diagnostics: how concentrated the outputs are, and how often findings are called positive
    vc = pd.Series([text_tokens(x) for x in preds]).value_counts(normalize=True)
    nf = LABELS.index("No Finding")
    r["interval_start"] = float(np.mean([text_tokens(x).startswith("interval") for x in preds]))
    r["per_label_precision"] = {c: float((P[:, j] & GT[s][:, j]).sum() / max(P[:, j].sum(), 1)) for j, c in enumerate(LABELS)}
    r.update({"top1_share": float(vc.iloc[0]), "top5_share": float(vc.iloc[:5].sum()),
              "pos_findings_per_study": float(np.delete(P, nf, 1).sum(1).mean()),
              "gt_pos_findings_per_study": float(np.delete(GT[s], nf, 1).sum(1).mean()),
              "no_finding_rate": float(P[:, nf].mean()), "gt_no_finding_rate": float(GT[s][:, nf].mean())})
    return r, P

RES, PMAT = {}, {}
for s in ("val", "test"):
    # CheXagent wraps sentences in markdown bold; '**' is not removed by the PTB tokenizer, so strip it for NLG metrics
    r, P = score([TC[i]["impression"].replace("**", " ").strip() for i in ORDER[s]], s, TCHCLS[s])
    RES[("teacher", s)], PMAT[("teacher", s)] = r, P
for run in runs:
    summ = json.load(open(f"{RUN_DIR}/{run}/summary.json"))
    for s in ("val", "test"):
        pr = pd.read_csv(f"{RUN_DIR}/{run}/pred_{s}.csv", keep_default_na=False).set_index("idx")
        _miss = len(set(ORDER[s]) - set(pr.index))
        assert _miss == 0, f"{run}: pred_{s}.csv lacks {_miss} of {len(ORDER[s])} {s} studies (a smoke run or a different split?)"
        pr = pr.loc[ORDER[s]]
        r, P = score(pr.pred.astype(str).tolist(), s)
        r.update({"best_epoch": summ["best_epoch"], "train_min": summ["train_minutes"], "params_deploy_M": round(summ["params"]["deploy_total"] / 1e6, 1),
                  "hit_cap": float(pr.hit_cap.mean()), "mean_tok": float(pr.n_tok.mean()), "stopped_early": summ["stopped_early"]})
        RES[(run, s)], PMAT[(run, s)] = r, P

SHOW = ["BLEU-1", "BLEU-4", "ROUGE-L", "METEOR", "CIDEr", "CE14_F1", "CE5_F1", "CE14_F1_vs_CSV", "CE5_F1_vs_CSV", "CE14_F1_rrg",
        "distinct", "top1_share", "top5_share", "pos_findings_per_study", "gt_pos_findings_per_study", "no_finding_rate",
        "gt_no_finding_rate", "interval_start", "mean_tok", "hit_cap", "best_epoch", "train_min", "params_deploy_M"]
BASE = {"NN-retrieval 384 (val)": {"BLEU-1": 0.2030, "BLEU-4": 0.0273, "ROUGE-L": 0.1455, "METEOR": 0.0852, "CIDEr": 0.0832, "CE14_F1_vs_CSV": 0.390, "CE5_F1_vs_CSV": 0.258},
        "NN-retrieval 384 (test)": {"BLEU-1": 0.2128, "BLEU-4": 0.0312, "ROUGE-L": 0.1481, "METEOR": 0.0909, "CIDEr": 0.0993, "CE14_F1_vs_CSV": 0.412, "CE5_F1_vs_CSV": 0.312}}
TABLE = {}
for s in ("val", "test"):
    tab = pd.DataFrame({k[0]: {m: RES[k].get(m) for m in SHOW} for k in RES if k[1] == s}).T
    tab = pd.concat([tab, pd.DataFrame(BASE).T.loc[[b for b in BASE if s in b]]])
    TABLE[s] = tab
    print(f"\n===== {s.upper()} ({len(ORDER[s])} studies) | CE*_F1 = CheXbert(generated) vs CheXbert(GT impression), uncertain = negative =====")
    print(tab.astype(float).round(4).to_string())
    print("CE14_F1 95% CI:", {k[0]: [round(x, 3) for x in RES[k]["CE14_F1_95CI"]] for k in RES if k[1] == s})

REF = "A0_ce" if "A0_ce" in runs else None
PAIRED = {}
if REF:
    print("\npaired bootstrap on TEST, CE14_F1(run) - CE14_F1(A0_ce), 95% CI:")
    for run in runs:
        if run != REF:
            d, ci = paired_boot(PMAT[(REF, "test")], PMAT[(run, "test")], GT["test"], LABELS)
            PAIRED[run] = {"mean": d, "ci95": ci, "significant": ci[0] > 0 or ci[1] < 0}
            print(f"  {run:16s} {d:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]" + ("  *" if PAIRED[run]["significant"] else ""))
# seed spread of the CE-only model: how much of a run-to-run difference is just training randomness
SEEDS = [r for r in ("A0_ce", "A0s1_ce", "A0s2_ce") if r in runs]
if len(SEEDS) > 1:
    v = np.array([RES[(r, "test")]["CE14_F1"] for r in SEEDS]); v5 = np.array([RES[(r, "test")]["CE5_F1"] for r in SEEDS])
    print(f"\nCE-only seed spread on TEST ({SEEDS}): CE14-F1 {np.round(v, 4).tolist()} mean {v.mean():.4f} sd {v.std(ddof=1):.4f} range {v.max() - v.min():.4f} | "
          f"CE5-F1 {np.round(v5, 4).tolist()} sd {v5.std(ddof=1):.4f}")
    SEED_SPREAD = {"runs": SEEDS, "ce14": v.tolist(), "ce14_sd": float(v.std(ddof=1)), "ce5": v5.tolist()}
    for r in SEEDS[1:]:
        d, ci = paired_boot(PMAT[("A0_ce", "test")], PMAT[(r, "test")], GT["test"], LABELS)
        print(f"  {r} - A0_ce (same config, different seed): {d:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]")
else:
    SEED_SPREAD = None

# clinical head (runs trained with gamma > 0): does the encoder detect findings that the generated text omits?
# per label, on TEST vs CheXbert(GT impression): head AUC, head F1 at a threshold chosen on VAL (max F1), and F1 of the generated text
def _auc(y, s):
    o = np.argsort(s); r = np.empty(len(s)); r[o] = np.arange(1, len(s) + 1); n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / max(n1 * n0, 1)) if n1 and n0 else float("nan")
def _f1(y, p):
    tp = (p & y).sum(); return float(2 * tp / max(2 * tp + (p & ~y).sum() + (~p & y).sum(), 1))
CLINHEAD = {}
for run in runs:
    pv = pd.read_csv(f"{RUN_DIR}/{run}/pred_val.csv", keep_default_na=False).set_index("idx").loc[ORDER["val"]]
    pt = pd.read_csv(f"{RUN_DIR}/{run}/pred_test.csv", keep_default_na=False).set_index("idx").loc[ORDER["test"]]
    if f"clinhead_{LABELS[0]}" not in pt.columns:
        continue
    rows = []
    for j, c in enumerate(LABELS):
        yv, yt = GT["val"][:, j].astype(bool), GT["test"][:, j].astype(bool)
        sv, st = pv[f"clinhead_{c}"].values.astype(float), pt[f"clinhead_{c}"].values.astype(float)
        grid = np.unique(np.quantile(sv, np.linspace(0.5, 0.995, 60)))
        thr = max(grid, key=lambda t: _f1(yv, sv >= t)) if yv.any() else 0.5
        rows.append({"label": c, "test_pos": int(yt.sum()), "head_AUC": _auc(yt, st), "head_F1_valthr": _f1(yt, st >= thr), "thr": float(thr),
                     "head_calls": int((st >= thr).sum()), "text_F1": _f1(yt, PMAT[(run, "test")][:, j].astype(bool)),
                     "text_calls": int(PMAT[(run, "test")][:, j].sum())})
    T = pd.DataFrame(rows).set_index("label")
    Ph = np.stack([pt[f"clinhead_{c}"].values >= T.loc[c, "thr"] for c in LABELS], 1).astype(np.int8)
    CLINHEAD[run] = {"per_label": T.to_dict(orient="index"), "head_CE14_F1": micro_f1(Ph, GT["test"]), "head_CE5_F1": micro_f1(Ph, GT["test"], FIVE)}
    print(f"\nclinical head of {run} on TEST (labels: CheXbert on GT impression; head threshold per label = max F1 on VAL):")
    print(T.round(3).to_string())
    print(f"  head as a classifier: CE14-F1 {CLINHEAD[run]['head_CE14_F1']:.4f}, CE5-F1 {CLINHEAD[run]['head_CE5_F1']:.4f} "
          f"vs the same run's generated text CE14-F1 {RES[(run, 'test')]['CE14_F1']:.4f}, CE5-F1 {RES[(run, 'test')]['CE5_F1']:.4f}")
# A10 diagnosis prompts: same model decoded with head / oracle (CheXbert of GT impression) / all-negative prompts, on TEST
PROMPT, A10_DECISION = {}, None
for run in runs:
    xf = {m: f"{RUN_DIR}/{run}/pred_test_{m}.csv" for m in ("oracle", "neg")}
    if not all(os.path.exists(p) for p in xf.values()):
        continue
    nf_ = LABELS.index("No Finding")
    PROMPT[run] = {"head": {"CE14_F1": RES[(run, "test")]["CE14_F1"], "CE5_F1": RES[(run, "test")]["CE5_F1"],
                            "pos_findings_per_study": RES[(run, "test")]["pos_findings_per_study"], "hit_cap": RES[(run, "test")]["hit_cap"]}}
    for m, p in xf.items():
        pr = pd.read_csv(p, keep_default_na=False).set_index("idx").loc[ORDER["test"]]
        P = CXB.binary(cxb.classes(pr.pred.astype(str).tolist()), LABELS, "pos_only")
        PROMPT[run][m] = {"CE14_F1": micro_f1(P, GT["test"]), "CE5_F1": micro_f1(P, GT["test"], FIVE),
                          "pos_findings_per_study": float(np.delete(P, nf_, 1).sum(1).mean()), "hit_cap": float(pr.hit_cap.mean())}
    print(f"\n{run}: TEST decodes with different prompts (head = the real model; oracle = CheXbert(GT impression) labels; neg = all not positive):")
    print(pd.DataFrame(PROMPT[run]).T.round(4).to_string())
A10S = [r for r in runs if r.startswith("A10")]
if A10S and REF:
    chk = {r: {"ci_low": PAIRED[r]["ci95"][0], "hit_cap": RES[(r, "test")]["hit_cap"]} for r in A10S}
    ok = len(A10S) >= 2 and all(v["ci_low"] > 0 and v["hit_cap"] <= 0.10 for v in chk.values())
    A10_DECISION = {"rule": "adopt if for both seeds the paired CE14-F1 CI (A10 - A0_ce) is entirely above 0 and hit_cap <= 10%",
                    "per_run": chk, "adopt": bool(ok)}
    print(f"\nPRE-REGISTERED RULE (A10): {A10_DECISION['rule']}\n  {chk}\n  -> {'ADOPT diagnosis prompts' if ok else 'NOT ADOPTED (keep CE-only A0)'}"
          + ("" if len(A10S) >= 2 else "  [only one A10 seed present]"))
# ---- v7 sequence-level KD: pre-registered rule, per method, both seeds
SEQKD_DECISION = {}
CE5_BAR = 0.226          # CE-only seed mean 0.188 + 3 sd (sd 0.0125), fixed before the v7 runs
for meth, pre in (("S1 teacher text -> GT fine-tune", "S1"), ("S2 mixed targets", "S2")):
    rr = [r for r in runs if r.startswith(pre)]
    if not rr or not REF:
        continue
    chk = {r: {"ci_low": PAIRED[r]["ci95"][0], "CE14_F1": RES[(r, "test")]["CE14_F1"], "CE5_F1": RES[(r, "test")]["CE5_F1"],
               "hit_cap": RES[(r, "test")]["hit_cap"], "pos_per_study": RES[(r, "test")]["pos_findings_per_study"],
               "interval_start": RES[(r, "test")]["interval_start"]} for r in rr}
    ok = len(rr) >= 2 and all(v["ci_low"] > -0.02 and v["CE5_F1"] > CE5_BAR and v["hit_cap"] <= 0.10 for v in chk.values())
    SEQKD_DECISION[pre] = {"method": meth, "per_run": chk, "adopt": bool(ok), "mean_CE5": float(np.mean([v["CE5_F1"] for v in chk.values()]))}
    print(f"\nPRE-REGISTERED RULE ({meth}): both seeds need paired CE14 CI lower bound > -0.02 vs A0, CE5-F1 > {CE5_BAR}, hit_cap <= 10%")
    print(pd.DataFrame(chk).T.round(4).to_string())
    print("  -> " + ("ADOPT" if ok else "NOT ADOPTED") + ("" if len(rr) >= 2 else f"  [only {len(rr)} seed present]"))
if sum(v["adopt"] for v in SEQKD_DECISION.values()) == 2:
    _w = max(SEQKD_DECISION, key=lambda k: SEQKD_DECISION[k]["mean_CE5"])
    print(f"both methods pass -> choose higher mean CE5: {_w}")
print("\nper-label precision on TEST (five common findings; teacher-hallucination check):")
print(pd.DataFrame({k[0]: {c: RES[k]["per_label_precision"][c] for c in FIVE} for k in RES if k[1] == "test"}).round(3).to_string())
print("\nper-label F1 on TEST:")
print(pd.DataFrame({k[0]: RES[k]["per_label_F1"] for k in RES if k[1] == "test"}).round(3).to_string())
for s in ("val", "test"):
    TABLE[s].to_csv(f"{WORK}/ablation_table_{s}.csv")
json.dump({"version": NOTEBOOK_VERSION, "mode": MODE, "run_set": RUN_SET, "connector": CONNECTOR,
           "results": {f"{k[0]}|{k[1]}": v for k, v in RES.items()}, "paired_vs_A0_test": PAIRED, "status": STATUS,
           "seed_spread": SEED_SPREAD, "clinical_head": CLINHEAD, "diag_prompt": PROMPT, "a10_decision": A10_DECISION, "seqkd_decision": SEQKD_DECISION, "max_len": MAX_LEN, "loss_weights": dict(beta=BETA, tau=TAU, delta=DELTA, gamma=GAMMA), "feature_target_diag": FEAT_DIAG},
          open(f"{WORK}/distill_results.json", "w"), indent=1, default=float)

In [ ]:
# ---- training curves and qualitative examples (for the report)
import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for run in runs:
    cv = [json.loads(l) for l in open(f"{RUN_DIR}/{run}/curve.jsonl")]
    ax[0].plot([c["epoch"] for c in cv], [c["val_ce"] for c in cv], marker="o", label=run)
    tr = [c for c in cv if "train_ce" in c]
    ax[1].plot([c["epoch"] for c in tr], [c["train_ce"] for c in tr], marker="o", label=run)
    ev = [c for c in cv if "val_ce14_f1" in c]
    ax[2].plot([c["epoch"] for c in ev], [c["val_ce14_f1"] for c in ev], marker="o", label=run)
for a, t in zip(ax, ["val CE (GT impression)", "train CE", "val CheXbert-14 micro-F1 (greedy)"]):
    a.set_title(t); a.set_xlabel("epoch"); a.grid(alpha=.3)
ax[2].axhline(RES[("teacher", "val")]["CE14_F1"], color="k", ls="--", lw=1, label="teacher (val)")
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/training_curves.png", dpi=130)
print("saved training_curves.png")

# per-term weighted training loss and gradient norm (all trainable params), per epoch: does one term dominate the update?
rows = []
for run in runs:
    for c in (json.loads(l) for l in open(f"{RUN_DIR}/{run}/curve.jsonl")):
        if "train_total" in c:
            rows.append({"run": run, "epoch": c["epoch"], **{k: c[k] for k in c if k.startswith("gradnorm_")},
                         **{k: c[k] for k in ("train_ce", "train_kd", "train_kd_kept", "train_clin", "train_feat") if k in c}})
GN = pd.DataFrame(rows)
if len(GN):
    print("\nper-term gradient norm (weighted term, averaged over probes) and unweighted train loss, by epoch:")
    print(GN.round(4).to_string(index=False))
    GN.to_csv(f"{WORK}/grad_norms.csv", index=False)

rs = np.random.RandomState(1)
te = SPL["test"]
nf = te["y_no_finding"].values == 1
pick = list(rs.choice(np.where(nf)[0], min(3, nf.sum()), replace=False)) + list(rs.choice(np.where(~nf)[0], min(5, (~nf).sum()), replace=False))
ex = pd.DataFrame({"idx": te.idx.values[pick], "GT": te.impression_final.values[pick],
                   "teacher": [TC[int(i)]["impression"] for i in te.idx.values[pick]]})
for run in runs:
    pr = pd.read_csv(f"{RUN_DIR}/{run}/pred_test.csv", keep_default_na=False).set_index("idx")
    ex[run] = pr.loc[ex.idx, "pred"].values
ex.to_csv(f"{WORK}/examples_test.csv", index=False)
for r in ex.itertuples(index=False):
    print(f"\nidx {r.idx}")
    for c, v in zip(ex.columns[1:], r[1:]):
        print(f"  {c:16s}: {str(v)[:220]}")
print("\nSMOKE OK - set MODE='train', commit on GPU T4 x2" if smoke else f"\nDISTILL {RUN_SET} DONE -> distill_results.json, ablation_table_*.csv, training_curves.png, examples_test.csv")